# LJ4 — A camera that never lies
### simulating the world for the fusion layer: golden scenarios vs the production temporal stack

*MVM Lab Journal — 4 of a series. Prev: LJ3 (Why walking people multiply). Next: LJ5 (Weighing
the experts).*

> The pose model is honest now (LJ1), the transitions have physics (LJ2), the tracker no longer
> multiplies walkers (LJ3) — each layer measured **in isolation**. But the product ships their
> *composition*: tracker → activities → FSM → anomaly → journal. That composition has only ever
> been judged on a live camera, where nobody knows what actually happened, so nobody can say
> "the journal lied". This journal builds a camera that cannot lie: scenarios whose truth is
> known **by construction** (from CMU mocap), injected *after* the pixel detectors, driven
> through bit-faithful ports of the production stack. Three questions: does the stack reproduce
> clean scenarios exactly? what breaks it under realistic detector noise — and which layer is
> guilty? and does the fall alarm actually tell a fall from lying down?

## 1. Context — the layers are measured; the composition is folklore

The live product exhibited two composite failures that no unit test ever caught: a walking person
fragmenting into phantom tracks fired a false `people_together` (two people who never existed), and
the fall alarm silently depended on details nobody had measured. LJ3 fixed the fragmentation at its
cause — but the *event level* (what the journal tells an LLM consumer) remained untested, because
on a live camera there is no ground truth to score against.

**The simulator's honest boundary:** we inject synthetic observations (bbox + posture + wrist
points + metric 3D) *after* the detectors. The pixel layer is explicitly out of scope — LJ1 and
LJ3 measured it. Everything above it runs for real, as byte-faithful Python ports of the Rust:
`infer_activities` (raw cross-group inference), the Activity FSM (enter/exit dwell, cooling,
churn stitching), the anomaly scorer (noisy-OR + decaying peak), the posture stabilizer (LJ2), and
the LJ3 tracker port. Each port self-checks against the Rust unit-test scenarios below — a port
that drifts is a stop-ship.

**Scenarios** (compiled from CMU mocap in world 3D, pelvis-continuous seams, eye-level camera):

| group | story | instances | the contract |
|---|---|---|---|
| G1 | walked in → sat → works at a keyboard | 3 | one `typing` episode, right timing; quiet anomaly |
| G2 | stood up → left the room | 3 | `typing` ends; track disappears; quiet |
| G3 | lay down / got up (controlled floor) | 10 | **anomaly stays silent** |
| G4 | fell | 6 | **anomaly fires** |
| G5 | two people met | 2 | one `people_together`, no id swap |
| G6 | a lone walker (LJ3 regression) | 4 | no events at all, one track |

Two compiler lessons paid for in blood (both were caught by the FLOOR run, not by eyeballs): the
BVH settling frame injects a fake `standing→lying` edge at t=0 (a phantom fall!), and expected
events must be derived from **what the injected data contains**, not what the author intended —
a walking-in person's wrists hang at seated-keyboard height, so 2D contact (production semantics,
no depth) honestly starts `typing` during the final approach steps. The production stack behaves
identically; the scenario truth now says so.

## 2. Hypotheses — failure criteria named up front

**S1 (the gate).** On clean injection (FLOOR) the production stack reproduces every scenario's
events with precision = recall = 1.0 within dwell tolerances and zero phantom events. Under
realistic detector stress (bursty occlusion, bbox jitter, intermittent face, posture flips) the
degradation is *measurable and attributable* to a layer via paired ablations.

> **Failure criterion.** If FLOOR already disagrees, it's either a stack defect (publish it — the
> simulator exists for exactly this), an unusable scenario truth (fix the compiler, not the
> tracker), or a broken port (stop-ship). No tuning until attribution.

**S2.** The production fall rule (*was standing within 3 s → lying now*, keyed **per track id**,
no velocity) false-alarms on fast controlled lie-downs and/or loses falls under stress; the LJ2
physics rule E2 (*reached floor AND peak pelvis descent ≥ 1.13 m/s*) separates them cleanly on the
same runs.

> **Failure criterion.** If the production rule behaves on both G3 and G4 — E2 stays in the lab
> and the 15.6 contract change is cancelled. If E2's 60 fps threshold fails at the production
> 15 fps cadence, we say so and recalibrate or retreat. Both endings publishable.

**S3.** Half the raw-activity vocabulary (`typing`, `using_mouse`, `watching_tv`…) is an artifact
of eyeballed model play that **no data can supervise** — measured against OpenImages VRD
*instances* (human-labeled, per-image), not the triplet vocabulary.

> **Failure criterion.** If VRD instances are plentiful for the current labels, the "artifact"
> directive is refuted by data — the vocabulary stays, and we say so.

## 3. Experiment — the scenario artifact and the ported stack

`golden_scenarios.npz`: 28 instances, 5 277 person-rows @15 fps (the production human-chain
cadence). Per frame: true person id, bbox, kinematic posture (biomechanics labeller, *not* the
GRIP features), wrist points, camera-frame pelvis (3D `people_together` distance) and world pelvis
(fall physics). Expected events + tolerances were emitted by the compiler — the scenario file *is*
the ground truth. The same file (a 6-instance FLOOR subset as JSON) drives a `cargo test` golden
gate inside the Rust workspace, so CI pins today's contract while this journal measures it.

In [ ]:
# ---- locate the scenario artifact (Kaggle dataset first, repo fallback) ----
import os, glob, json, collections
import numpy as np

def _find(*names):
    roots = ["/kaggle/input/mvm-golden-scenarios",
             os.path.join("..","..","..","spike_pose","data_probe"),
             os.path.join("..","..","spike_pose","data_probe")]
    for r in roots:
        for n in names:
            if os.path.exists(os.path.join(r, n)): return os.path.join(r, n)
    for n in names:
        hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        if hits: return hits[0]
    have = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input"
    raise FileNotFoundError((names, "mounted:", have))

GOLD = _find("golden_scenarios.npz")
_d = np.load(GOLD, allow_pickle=True)
_meta = json.loads(str(_d["meta_json"]))
print(f"{len(_meta)} instances:",
      dict(collections.Counter(m["group"] for m in _meta.values())),
      f"| {len(_d['inst'])} person-rows @15fps")

### 3.1 The fusion stack, ported byte-for-byte — and self-checked against the Rust unit tests

The next two cells are **the lab's actual source files**, embedded by the notebook builder (not
hand-copied): `lj4_fusion.py` (raw activities + FSM + anomaly, constants quoted at their Rust
definitions) and `lj3_tracker.py` (the LJ3 association port). Each ends by replaying the Rust
unit-test scenarios — 39 for the fusion stack — as a live bit-faithfulness check.

In [ ]:
#!/usr/bin/env python3
"""Faithful Python port of the MVM fusion temporal stack (phase 15.4 lab bench).

Ported byte-for-byte from the Rust (constants quoted at their definitions):
  - infer_activities        mvm-cli/src/agents/fusion.rs:365 (+ helpers 189-355)
  - ActivityJournal (FSM)   mvm-core/src/activity.rs (enter/exit dwell, theta_lo(d),
                            max-dwell cap, churn stitching 11.C6, caps)
  - AnomalyScorer (v2)      mvm-core/src/anomaly.rs (noisy-OR + decaying peak)
  - FusionStep              the advance_journal frame-gating (fusion.rs:51) — the FSM
                            ticks once per NEW frame_id, never per loop iteration.

Same lab choice as LJ2's stabilizer and LJ3's tracker ports: run the production logic
in Python so golden scenarios sweep cheaply with bootstrap CIs; the Rust port of
whatever wins (+ bit-exact golden tests) is 15.6.

Honest scope cuts (each is the prod fallback branch, not an approximation):
  - No instance masks in the bench → contact = bbox test (`point_near_bbox`), the exact
    branch prod takes when masks are off / absent (fusion.rs:241-246). Mask-contact and
    mask-centroid paths are NOT ported; scenarios must not carry masks.
  - No depth maps → `anchor_objects` is not ported; scenarios inject `position_3d`
    directly (the simulator's contract: injection AFTER the detectors).
  - Anti-pet gate (advance_journal's animal filter) not ported — scenarios carry no
    animal detections.

Self-check: `python3 lj4_fusion.py` replays the Rust unit-test scenarios from
activity.rs / anomaly.rs / fusion.rs and must print OK. A drifting port is a stop-ship.
"""
from dataclasses import dataclass, field
from typing import Optional

# ── production constants ───────────────────────────────────────────────────────
# activity.rs
ACTIVITY_ENTER_DWELL = 3        # ln 30
ACTIVITY_EXIT_DWELL = 5         # ln 34
ACTIVITY_EXIT_DWELL_YOUNG = 8   # ln 38
ACTIVITY_EXIT_DWELL_OLD = 4     # ln 42
ACTIVITY_D_MIN_S = 2.0          # ln 44
ACTIVITY_D_TYP_S = 600.0        # ln 47
ACTIVITY_CONF_LAMBDA = 0.7      # ln 49
ACTIVITY_INSTANCE_CAP = 64      # ln 53
ACTIVITY_EVENT_CAP = 256        # ln 55
# anomaly.rs
EXPECTED_OBJECTS = {
    "chair", "couch", "bed", "dining table", "tv", "laptop", "mouse", "remote",
    "keyboard", "cell phone", "book", "clock", "vase", "potted plant", "cup",
    "bottle", "bowl", "fork", "spoon", "microwave", "oven", "toaster", "sink",
    "refrigerator",
}                               # ln 25
ANOMALY_OBJECT_WEIGHT = 0.25    # ln 33
ANOMALY_POSE_OOD = 0.3          # ln 35
ANOMALY_FALL = 0.6              # ln 37
ANOMALY_FALL_WINDOW_S = 3.0     # ln 39
ANOMALY_DECAY = 0.9             # ln 41
STANDING_MEMORY_CAP = 64        # ln 43
# fusion.rs
ACTIVITY_NEAR_M = 1.5           # ln 192
CONTACT_MARGIN = 0.03           # ln 195
FRAME_EDGE_MARGIN = 0.02        # ln 198
WORKSTATION_MARGIN = 0.10       # ln 201
WORKSTATION_CLASSES = ("dining table", "laptop", "keyboard", "mouse", "tv", "book")  # ln 208


def interaction_label(cls):
    """fusion.rs:213 — activity label for a person↔object interaction, else None."""
    return {
        "keyboard": "typing", "laptop": "typing",
        "mouse": "using_mouse",
        "cell phone": "using_phone",
        "cup": "holding_drink", "bottle": "holding_drink",
        "wine glass": "holding_drink", "bowl": "holding_drink",
        "book": "reading",
        "remote": "watching_tv", "tv": "watching_tv",
    }.get(cls)


def max_dwell_s(label):
    """activity.rs:136 — bounded-duration cap per label."""
    return {"holding_drink": 900.0, "using_phone": 1800.0}.get(label, 3600.0)


# ── injected observations (the simulator's post-detector contract) ─────────────

@dataclass
class Person:
    """PersonState subset the fusion stack reads (world.rs). bbox normalized
    [x1,y1,x2,y2]; position_3d (x,y,z) metric or None; posture (label, conf) or
    None; wrists = [[x,y],...]; hands = [[[x,y],...] per hand]."""
    track_id: int
    bbox: tuple = (0.3, 0.2, 0.5, 0.7)
    position_3d: Optional[tuple] = None
    posture: Optional[tuple] = None
    plausibility: Optional[str] = None
    wrists: list = field(default_factory=list)
    hands: list = field(default_factory=list)
    left_gesture: Optional[str] = None
    right_gesture: Optional[str] = None
    body_coverage: Optional[str] = None
    torso_attitude: Optional[str] = None
    motion_state: Optional[str] = None


@dataclass
class Obj:
    """ObjectState subset (id ≥ 10000 in prod id-space)."""
    id: int
    cls: str
    bbox: tuple
    confidence: float = 0.8


@dataclass
class Activity:
    label: str
    confidence: float
    participants: list


# ── raw cross-group inference (fusion.rs:365) ──────────────────────────────────

def _point_near_bbox(pt, b):
    """fusion.rs:226 — normalised point within b expanded by CONTACT_MARGIN."""
    return (b[0] - CONTACT_MARGIN <= pt[0] <= b[2] + CONTACT_MARGIN
            and b[1] - CONTACT_MARGIN <= pt[1] <= b[3] + CONTACT_MARGIN)


def _contact_centre(o):
    """fusion.rs:251 — bbox centre (mask centroid path out of bench scope)."""
    return ((o.bbox[0] + o.bbox[2]) * 0.5, (o.bbox[1] + o.bbox[3]) * 0.5)


class _ContactClaim:
    """fusion.rs:271 — one person's claim on a contested object."""

    def __init__(self, track_id, hand_touch, reaching_wrist_only, dist2):
        self.track_id = track_id
        self.hand_touch = hand_touch
        self.reaching_wrist_only = reaching_wrist_only
        self.dist2 = dist2

    def beats(self, other):
        if self.hand_touch != other.hand_touch:
            return self.hand_touch
        if self.reaching_wrist_only != other.reaching_wrist_only:
            return not self.reaching_wrist_only
        if self.dist2 != other.dist2:
            return self.dist2 < other.dist2
        return self.track_id < other.track_id


def _contact_claim(p, o, centre):
    """fusion.rs:295 — this person's claim on the object, or None."""
    hand_touch = wrist_touch = False
    dist2 = float("inf")

    def consider(pt, is_hand):
        nonlocal hand_touch, wrist_touch, dist2
        if _point_near_bbox(pt, o.bbox):
            if is_hand:
                hand_touch = True
            else:
                wrist_touch = True
            dx, dy = pt[0] - centre[0], pt[1] - centre[1]
            dist2 = min(dist2, dx * dx + dy * dy)

    for pt in p.wrists:
        consider(pt, False)
    for hand in p.hands:
        for pt in hand:
            consider(pt, True)
    if not hand_touch and not wrist_touch:
        return None
    reaching = p.left_gesture == "reaching" or p.right_gesture == "reaching"
    return _ContactClaim(p.track_id, hand_touch, not hand_touch and reaching, dist2)


def _dist3(a, b):
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2 + (a[2] - b[2]) ** 2) ** 0.5


def _bbox_adjacent(a, b, margin):
    """fusion.rs:336 — a expanded by margin intersects b."""
    return (a[0] - margin < b[2] and b[0] < a[2] + margin
            and a[1] - margin < b[3] and b[1] < a[3] + margin)


def _bbox_at_frame_edge(b):
    """fusion.rs:342."""
    return (b[0] <= FRAME_EDGE_MARGIN or b[1] <= FRAME_EDGE_MARGIN
            or b[2] >= 1.0 - FRAME_EDGE_MARGIN or b[3] >= 1.0 - FRAME_EDGE_MARGIN)


def _upper_upright_stationary(p):
    """fusion.rs:351."""
    return (p.body_coverage == "upper" and p.torso_attitude == "upright"
            and p.motion_state != "moving")


def infer_activities(people, objects):
    """fusion.rs:365 — raw per-tick activities, pure arithmetic."""
    acts = []

    # person ↔ person proximity (needs both torso-3D positions).
    for i in range(len(people)):
        for j in range(i + 1, len(people)):
            pa, pb = people[i].position_3d, people[j].position_3d
            if pa is not None and pb is not None:
                d = _dist3(pa, pb)
                if d < ACTIVITY_NEAR_M:
                    acts.append(Activity(
                        "people_together",
                        min(max(1.0 - d / ACTIVITY_NEAR_M, 0.0), 1.0),
                        [people[i].track_id, people[j].track_id]))

    # person ↔ interactive object by hand/wrist contact, object-major (11.A2).
    for o in objects:
        label = interaction_label(o.cls)
        if label is None:
            continue
        centre = _contact_centre(o)
        best = None
        for p in people:
            claim = _contact_claim(p, o, centre)
            if claim is not None and (best is None or claim.beats(best)):
                best = claim
        if best is not None:
            acts.append(Activity(label, min(max(o.confidence, 0.0), 1.0),
                                 [best.track_id, o.id]))

    # Occlusion contexts (occlusion-aware posture).
    for p in people:
        if _upper_upright_stationary(p):
            for o in objects:
                if (o.cls in WORKSTATION_CLASSES
                        and _bbox_adjacent(p.bbox, o.bbox, WORKSTATION_MARGIN)):
                    acts.append(Activity("seated_at_workstation",
                                         min(max(o.confidence, 0.0), 1.0),
                                         [p.track_id, o.id]))
                    break
        if (p.body_coverage is not None and p.body_coverage != "full"
                and _bbox_at_frame_edge(p.bbox)):
            acts.append(Activity("partially_in_frame", 1.0, [p.track_id]))

    return acts


# ── Activity FSM + journal (activity.rs) ───────────────────────────────────────

def _churn_class(pid):
    """activity.rs:90 — None reserved (0-9), 0 ephemeral person, 1 object."""
    if pid < 10:
        return None
    return 0 if pid < 10_000 else 1


def _one_churn_apart(a, b):
    """activity.rs:105 — sorted sets differ in exactly one same-class churnable id."""
    a_only = [x for x in a if x not in b]
    b_only = [x for x in b if x not in a]
    return (len(a_only) == 1 and len(b_only) == 1
            and _churn_class(a_only[0]) is not None
            and _churn_class(a_only[0]) == _churn_class(b_only[0]))


CANDIDATE, ACTIVE, COOLING, ENDED = "candidate", "active", "cooling", "ended"


class ActivityInstance:
    def __init__(self, key, phase, enter_streak, started_at, first_seen_frame,
                 last_seen, confidence, peak_confidence):
        self.key = key                      # (label, tuple(sorted participants))
        self.phase = phase
        self.enter_streak = enter_streak
        self.absent_streak = 0
        self.started_at = started_at
        self.first_seen_frame = first_seen_frame
        self.last_seen = last_seen
        self.confidence = confidence
        self.peak_confidence = peak_confidence


@dataclass
class ActivityEvent:
    label: str
    participants: list
    started_at: float
    ended_at: float
    duration_s: float
    peak_confidence: float
    stable_participants: bool


def _stable_participants(participants):
    """activity.rs:77 — reserved (0-9) or object (≥10000) ids only."""
    return all(pid < 10 or pid >= 10_000 for pid in participants)


def _event_from(inst):
    return ActivityEvent(
        label=inst.key[0], participants=list(inst.key[1]),
        started_at=inst.started_at, ended_at=inst.last_seen,
        duration_s=inst.last_seen - inst.started_at,
        peak_confidence=inst.peak_confidence,
        stable_participants=_stable_participants(inst.key[1]))


class ActivityJournal:
    """activity.rs:220 — temporal debounce + event journal, stitching opt-in.

    The exit-dwell knobs are exposed for the LJ4 sweep (lab pattern, like
    lj3_tracker's gates); the defaults ARE the production constants."""

    def __init__(self, stitch_churn=False, exit_dwell=ACTIVITY_EXIT_DWELL,
                 exit_young=ACTIVITY_EXIT_DWELL_YOUNG, exit_old=ACTIVITY_EXIT_DWELL_OLD):
        self.instances = []     # sorted by key — deterministic emission order
        self.completed = []
        self.stitch_churn = stitch_churn
        self.exit_dwell, self.exit_young, self.exit_old = exit_dwell, exit_young, exit_old

    def update(self, raw, t_secs, frame_id):
        # Present keys → strongest observed confidence this tick (sorted).
        present = {}
        for a in raw:
            key = (a.label, tuple(sorted(a.participants)))
            present[key] = max(present.get(key, 0.0), a.confidence)
        present = sorted(present.items())

        # Advance existing instances (in stored sorted order).
        for inst in self.instances:
            obs = None
            for i, (k, conf) in enumerate(present):
                if k == inst.key:
                    obs = conf
                    present.pop(i)
                    break
            if obs is not None:
                inst.absent_streak = 0
                inst.last_seen = t_secs
                inst.confidence = (ACTIVITY_CONF_LAMBDA * inst.confidence
                                   + (1.0 - ACTIVITY_CONF_LAMBDA) * obs)
                inst.peak_confidence = max(inst.peak_confidence, inst.confidence)
                if inst.phase == CANDIDATE:
                    inst.enter_streak += 1
                    if inst.enter_streak >= ACTIVITY_ENTER_DWELL:
                        inst.phase = ACTIVE
                elif inst.phase == COOLING:
                    inst.phase = ACTIVE
                # Bounded-duration cap (11.C2).
                if (inst.phase == ACTIVE
                        and inst.last_seen - inst.started_at >= max_dwell_s(inst.key[0])):
                    inst.phase = ENDED
                    self.completed.append(_event_from(inst))
            else:
                if inst.phase == CANDIDATE:
                    inst.phase = ENDED  # 1-frame spike never reaches the journal
                elif inst.phase in (ACTIVE, COOLING):
                    inst.phase = COOLING
                    inst.absent_streak += 1
                    d = inst.last_seen - inst.started_at
                    if d < ACTIVITY_D_MIN_S:
                        exit_dwell = self.exit_young
                    elif d > ACTIVITY_D_TYP_S:
                        exit_dwell = self.exit_old
                    else:
                        exit_dwell = self.exit_dwell
                    if inst.absent_streak >= exit_dwell:
                        inst.phase = ENDED
                        self.completed.append(_event_from(inst))

        # Reap ended/lapsed.
        self.instances = [i for i in self.instances if i.phase != ENDED]

        # Keys still unmatched are newborn candidates (bounded).
        for key, conf in present:
            if len(self.instances) >= ACTIVITY_INSTANCE_CAP:
                break
            src = None
            if self.stitch_churn and len(key[1]) >= 2:
                src = self._take_stitch_source(key)
            if src is not None:
                inst = ActivityInstance(key, ACTIVE, ACTIVITY_ENTER_DWELL,
                                        src.started_at, src.first_seen_frame,
                                        t_secs, conf, max(src.peak_confidence, conf))
            else:
                inst = ActivityInstance(key, CANDIDATE, 1, t_secs, frame_id,
                                        t_secs, conf, conf)
            # Insert keeping sort order.
            at = 0
            while at < len(self.instances) and self.instances[at].key < inst.key:
                at += 1
            self.instances.insert(at, inst)

        # Bounded event backlog even if the consumer never drains.
        if len(self.completed) > ACTIVITY_EVENT_CAP:
            del self.completed[:len(self.completed) - ACTIVITY_EVENT_CAP]

    def _take_stitch_source(self, newborn):
        """activity.rs:383 — remove + return the cooling instance this newborn
        continues (same label, one churnable participant apart)."""
        for i, inst in enumerate(self.instances):
            if (inst.phase == COOLING and inst.key[0] == newborn[0]
                    and len(inst.key[1]) == len(newborn[1])
                    and _one_churn_apart(inst.key[1], newborn[1])):
                return self.instances.pop(i)
        return None

    def active(self):
        """Stabilized activities to emit: Active + Cooling (grace-bridged)."""
        return [i for i in self.instances if i.phase in (ACTIVE, COOLING)]

    def drain_events(self):
        ev, self.completed = self.completed, []
        return ev


# ── Anomaly v2 scorer (anomaly.rs) ─────────────────────────────────────────────

class AnomalyScorer:
    """anomaly.rs:48 — noisy-OR of unexpected-object / pose-OOD / sudden-fall,
    published as a decaying peak."""

    def __init__(self):
        self.standing_seen = []     # [(track_id, t_secs last seen standing)]
        self.score = 0.0

    def update(self, people, objects, t_secs):
        components = []

        unexpected = sum(1 for o in objects if o.cls not in EXPECTED_OBJECTS)
        if unexpected > 0:
            components.append(min(unexpected * ANOMALY_OBJECT_WEIGHT, 1.0))

        for p in people:
            label = p.posture[0] if p.posture is not None else None
            if label == "standing":
                for e in self.standing_seen:
                    if e[0] == p.track_id:
                        e[1] = t_secs
                        break
                else:
                    if len(self.standing_seen) < STANDING_MEMORY_CAP:
                        self.standing_seen.append([p.track_id, t_secs])
            if label == "lying":
                recently_stood = any(
                    tid == p.track_id and t_secs - t <= ANOMALY_FALL_WINDOW_S
                    for tid, t in self.standing_seen)
                if recently_stood:
                    components.append(ANOMALY_FALL)
            if p.plausibility == "low":
                components.append(ANOMALY_POSE_OOD)

        # Forget stale standing memory (well past the window).
        self.standing_seen = [e for e in self.standing_seen
                              if t_secs - e[1] <= ANOMALY_FALL_WINDOW_S * 4.0]

        raw = 1.0
        for c in components:
            raw *= 1.0 - min(max(c, 0.0), 1.0)
        raw = 1.0 - raw
        self.score = max(raw, self.score * ANOMALY_DECAY)
        if self.score < 1e-3:
            self.score = 0.0
        return self.score


# ── FusionStep — the advance_journal frame-gating (fusion.rs:51) ───────────────

class FusionStep:
    """One world tick: raw inference every call, FSM/anomaly advanced once per
    NEW frame_id (the mesh loop spins faster than frames arrive). Stitching is
    ON as in the mesh FusionAgent (fusion.rs:40)."""

    def __init__(self, stitch_churn=True, journal_kw=None):
        self.journal = ActivityJournal(stitch_churn=stitch_churn, **(journal_kw or {}))
        self.anomaly = AnomalyScorer()
        self.last_frame = None

    def tick(self, people, objects, t_secs, frame_id):
        """Returns (activities, new_events, anomaly_score) — the stabilized set
        (Active+Cooling with EMA confidence), episodes completed this tick, and
        the decaying-peak anomaly score."""
        raw = infer_activities(people, objects)
        events = []
        if self.last_frame != frame_id:
            self.last_frame = frame_id
            self.journal.update(raw, t_secs, frame_id)
            events = self.journal.drain_events()
            self.anomaly_score = self.anomaly.update(people, objects, t_secs)
        activities = [Activity(i.key[0], i.confidence, list(i.key[1]))
                      for i in self.journal.active()]
        return activities, events, getattr(self, "anomaly_score", 0.0)


# ── self-check: replay the Rust unit-test scenarios ───────────────────────────

def _act(label, participants, conf=0.8):
    return Activity(label, conf, participants)


def _drive(j, raw, n, t0, f0):
    t, f = t0, f0
    for _ in range(n):
        j.update(raw, t, f)
        t += 0.1
        f += 1
    return t, f


def _check_fsm():
    typing = lambda: _act("typing", [0, 10001])

    # one_frame_spike_never_emits
    j = ActivityJournal()
    j.update([typing()], 0.0, 0)
    assert len(j.active()) == 0
    j.update([], 0.1, 1)
    j.update([typing()], 0.2, 2)
    j.update([], 0.3, 3)
    assert len(j.active()) == 0 and not j.drain_events()

    # enter_dwell_confirms_after_three_consecutive
    j = ActivityJournal()
    _drive(j, [typing()], 2, 0.0, 0)
    assert len(j.active()) == 0
    j.update([typing()], 0.2, 2)
    inst = j.active()
    assert len(inst) == 1 and inst[0].phase == ACTIVE
    assert inst[0].started_at == 0.0 and inst[0].first_seen_frame == 0

    # short_dropout_is_bridged_by_cooling
    j = ActivityJournal()
    t, f = _drive(j, [typing()], 3, 0.0, 0)
    t, f = _drive(j, [], 2, t, f)
    inst = j.active()
    assert len(inst) == 1 and inst[0].phase == COOLING
    j.update([typing()], t, f)
    inst = j.active()
    assert inst[0].phase == ACTIVE and inst[0].started_at == 0.0
    assert not j.drain_events()

    # exit_dwell_ends_with_honest_duration
    j = ActivityJournal()
    t, f = _drive(j, [typing()], 30, 0.0, 0)
    _drive(j, [], ACTIVITY_EXIT_DWELL, t, f)
    assert len(j.active()) == 0
    ev = j.drain_events()
    assert len(ev) == 1 and ev[0].label == "typing"
    assert ev[0].participants == [0, 10001] and ev[0].started_at == 0.0
    assert abs(ev[0].ended_at - 2.9) < 1e-6 and abs(ev[0].duration_s - 2.9) < 1e-6
    assert not j.drain_events()

    # confidence_is_ema_and_peak_tracked
    j = ActivityJournal()
    j.update([_act("typing", [0, 10001], 1.0)], 0.0, 0)
    j.update([_act("typing", [0, 10001], 0.0)], 0.1, 1)
    assert abs(j.instances[0].confidence - 0.7) < 1e-6
    assert abs(j.instances[0].peak_confidence - 1.0) < 1e-6

    # duplicate_keys_and_participant_order_collapse
    j = ActivityJournal()
    raw = [_act("typing", [10001, 0], 0.5), _act("typing", [0, 10001], 0.9)]
    _drive(j, raw, 3, 0.0, 0)
    assert len(j.active()) == 1

    # distinct_participant_sets_are_distinct_instances
    j = ActivityJournal()
    raw = [_act("typing", [0, 10001]), _act("typing", [1, 10001])]
    _drive(j, raw, 3, 0.0, 0)
    assert len(j.active()) == 2

    # emission_order_is_deterministic
    a, b = ActivityJournal(), ActivityJournal()
    _drive(a, [_act("typing", [0, 10001]), _act("reading", [1, 10002])], 3, 0.0, 0)
    _drive(b, [_act("reading", [1, 10002]), _act("typing", [0, 10001])], 3, 0.0, 0)
    assert [i.key for i in a.active()] == [i.key for i in b.active()]

    # participant_churn_ends_and_restarts (default: no stitching)
    j = ActivityJournal()
    t, f = _drive(j, [_act("typing", [10, 10001])], 6, 0.0, 0)
    _drive(j, [_act("typing", [11, 10001])], 8, t, f)
    active = j.active()
    assert len(active) == 1 and list(active[0].key[1]) == [11, 10001]
    ev = j.drain_events()
    assert len(ev) == 1 and ev[0].participants == [10, 10001]

    # stitching_bridges_ephemeral_churn_on_same_object
    j = ActivityJournal(stitch_churn=True)
    t, f = _drive(j, [_act("typing", [12, 10001])], 30, 0.0, 0)
    t, f = _drive(j, [_act("typing", [15, 10001])], 8, t, f)
    assert not j.drain_events(), "churn must not split the episode"
    inst = j.active()
    assert len(inst) == 1 and list(inst[0].key[1]) == [15, 10001]
    assert inst[0].started_at == 0.0
    _drive(j, [], ACTIVITY_EXIT_DWELL, t, f)
    ev = j.drain_events()
    assert len(ev) == 1 and ev[0].started_at == 0.0 and ev[0].duration_s > 3.0

    # stitching_does_not_bridge_across_different_objects
    j = ActivityJournal(stitch_churn=True)
    t, f = _drive(j, [_act("typing", [12, 10001])], 6, 0.0, 0)
    _drive(j, [_act("typing", [15, 10002])], 8, t, f)
    assert j.drain_events(), "two-participant churn → end+restart"

    # stitching_bridges_object_churn_under_stable_person
    j = ActivityJournal(stitch_churn=True)
    t, f = _drive(j, [_act("typing", [0, 10000])], 30, 0.0, 0)
    t, f = _drive(j, [_act("typing", [0, 10007])], 8, t, f)
    assert not j.drain_events()
    inst = j.active()
    assert len(inst) == 1 and list(inst[0].key[1]) == [0, 10007]
    assert inst[0].started_at == 0.0
    _drive(j, [], ACTIVITY_EXIT_DWELL, t, f)
    ev = j.drain_events()
    assert len(ev) == 1 and ev[0].started_at == 0.0 and ev[0].duration_s > 3.0

    # stitching_never_merges_two_gallery_people
    j = ActivityJournal(stitch_churn=True)
    t, f = _drive(j, [_act("typing", [0, 10001])], 6, 0.0, 0)
    _drive(j, [_act("typing", [5, 10001])], 8, t, f)
    assert j.drain_events(), "person 0 ≠ person 5 → honest end+restart"

    # stable_participants_reflects_id_spaces
    assert _stable_participants([0, 10001]) and not _stable_participants([42, 10001])
    j = ActivityJournal()
    t, f = _drive(j, [_act("typing", [0, 10001])], 3, 0.0, 0)
    _drive(j, [], ACTIVITY_EXIT_DWELL + 3, t, f)
    ev = j.drain_events()
    assert len(ev) == 1 and ev[0].stable_participants

    # max_dwell_caps_duration_and_restarts
    j = ActivityJournal()
    raw = [_act("typing", [0, 10001])]
    t = 0.0
    for fr in range(5000):
        j.update(raw, t, fr)
        t += 1.0
    ev = j.drain_events()
    assert ev, "cap must have force-ended at least once"
    assert all(e.duration_s <= max_dwell_s("typing") + 1.0 for e in ev)
    assert len(j.active()) == 1, "activity restarts after the forced end"

    # young_activity_survives_longer_gap
    j = ActivityJournal()
    t, f = _drive(j, [typing()], 3, 0.0, 0)
    t, f = _drive(j, [], ACTIVITY_EXIT_DWELL + 1, t, f)
    assert len(j.active()) == 1, "young instance still cooling"
    j.update([typing()], t, f)
    assert j.active()[0].started_at == 0.0 and not j.drain_events()

    # old_activity_exits_faster
    j = ActivityJournal()
    _, f = _drive(j, [typing()], 3, 0.0, 0)
    t = 100.0
    while t < ACTIVITY_D_TYP_S + 200.0:
        j.update([typing()], t, f)
        t += 100.0
        f += 1
    _drive(j, [], ACTIVITY_EXIT_DWELL_OLD, t, f)
    assert len(j.active()) == 0 and len(j.drain_events()) == 1

    # instance_cap_bounds_memory
    j = ActivityJournal()
    flood = [_act("typing", [i, 10001], 0.5) for i in range(2 * ACTIVITY_INSTANCE_CAP)]
    _drive(j, flood, 3, 0.0, 0)
    assert len(j.instances) <= ACTIVITY_INSTANCE_CAP

    # event_cap_bounds_undrained_backlog
    j = ActivityJournal()
    t, f = 0.0, 0
    for i in range(ACTIVITY_EVENT_CAP + 40):
        raw = [_act("typing", [i, 10001], 0.5)]
        for _ in range(ACTIVITY_ENTER_DWELL):
            j.update(raw, t, f)
            t += 0.1
            f += 1
        for _ in range(ACTIVITY_EXIT_DWELL):
            j.update([], t, f)
            t += 0.1
            f += 1
    assert len(j.completed) <= ACTIVITY_EVENT_CAP


def _check_anomaly():
    P = lambda tid, posture: Person(tid, posture=(posture, 0.9) if posture else None)
    O = lambda cls: Obj(10000, cls, (0.4, 0.4, 0.5, 0.5))

    # calm_scene_scores_zero
    s = AnomalyScorer()
    assert s.update([P(0, "standing")], [O("cup")], 0.0) == 0.0

    # unexpected_object_raises
    s = AnomalyScorer()
    assert abs(s.update([], [O("knife")], 0.0) - ANOMALY_OBJECT_WEIGHT) < 1e-6

    # sudden_fall_flags_but_longterm_lying_does_not
    s = AnomalyScorer()
    s.update([P(0, "standing")], [], 0.0)
    assert abs(s.update([P(0, "lying")], [], 1.0) - ANOMALY_FALL) < 1e-6
    s2 = AnomalyScorer()
    assert s2.update([P(0, "lying")], [], 0.0) == 0.0
    s3 = AnomalyScorer()
    s3.update([P(0, "standing")], [], 0.0)
    for t in range(1, 40):
        s3.update([P(0, "sitting")], [], float(t))
    assert s3.update([P(0, "lying")], [], 40.0) < 0.05

    # knife_plus_fall_scores_above_either_alone
    s = AnomalyScorer()
    s.update([P(0, "standing")], [], 0.0)
    combined = s.update([P(0, "lying")], [O("knife")], 1.0)
    expected = 1.0 - (1.0 - ANOMALY_FALL) * (1.0 - ANOMALY_OBJECT_WEIGHT)
    assert abs(combined - expected) < 1e-6
    assert ANOMALY_FALL < combined <= 1.0

    # pose_ood_contributes
    s = AnomalyScorer()
    p = P(0, "standing")
    p.plausibility = "low"
    assert abs(s.update([p], [], 0.0) - ANOMALY_POSE_OOD) < 1e-6

    # spike_decays_instead_of_vanishing
    s = AnomalyScorer()
    s.update([P(0, "standing")], [], 0.0)
    spike = s.update([P(0, "lying")], [], 1.0)
    nxt = s.update([], [], 2.0)
    assert abs(nxt - spike * ANOMALY_DECAY) < 1e-6
    last = nxt
    for t in range(3, 120):
        last = s.update([], [], float(t))
    assert last == 0.0


def _check_infer():
    def person(tid=10, z=1.0):
        return Person(tid, position_3d=(0.0, 0.0, z))

    def person_id(tid):
        return Person(tid, position_3d=None)

    # typing_when_wrist_on_keyboard
    p = person()
    p.wrists = [[0.40, 0.58]]
    kb = Obj(10001, "keyboard", (0.35, 0.55, 0.45, 0.62))
    acts = infer_activities([p], [kb])
    assert any(a.label == "typing" and a.participants == [10, 10001] for a in acts)

    # contested_object_goes_to_hand_not_reaching_wrist
    phone = Obj(10001, "cell phone", (0.5, 0.5, 0.6, 0.6))
    a = person_id(10)
    a.wrists = [[0.51, 0.55]]
    a.right_gesture = "reaching"
    b = person_id(11)
    b.hands = [[[0.58, 0.55]]]
    acts = infer_activities([a, b], [phone])
    phone_acts = [x for x in acts if x.label == "using_phone"]
    assert len(phone_acts) == 1 and phone_acts[0].participants == [11, 10001]

    # contested_object_goes_to_closest_wrist
    cup = Obj(10002, "cup", (0.4, 0.4, 0.6, 0.6))
    a = person_id(10)
    a.wrists = [[0.41, 0.5]]
    b = person_id(11)
    b.wrists = [[0.5, 0.5]]
    acts = infer_activities([a, b], [cup])
    cup_acts = [x for x in acts if x.label == "holding_drink"]
    assert len(cup_acts) == 1 and cup_acts[0].participants == [11, 10002]

    # uncontested_contact_still_fires (reaching alone must not silence it)
    a = person_id(10)
    a.wrists = [[0.41, 0.5]]
    a.right_gesture = "reaching"
    acts = infer_activities([a], [cup])
    assert any(x.label == "holding_drink" and x.participants == [10, 10002] for x in acts)

    # two_objects_two_people_attribute_independently
    kb1 = Obj(10001, "keyboard", (0.1, 0.5, 0.3, 0.6))
    kb2 = Obj(10002, "keyboard", (0.6, 0.5, 0.8, 0.6))
    a = person_id(10)
    a.wrists = [[0.2, 0.55]]
    b = person_id(11)
    b.wrists = [[0.7, 0.55]]
    acts = infer_activities([a, b], [kb1, kb2])
    assert any(x.label == "typing" and x.participants == [10, 10001] for x in acts)
    assert any(x.label == "typing" and x.participants == [11, 10002] for x in acts)

    # hand_landmark_triggers_object_use
    p = person()
    p.hands = [[[0.33, 0.57], [0.34, 0.56]]]
    mouse = Obj(10001, "mouse", (0.31, 0.54, 0.36, 0.60))
    assert any(a.label == "using_mouse" for a in infer_activities([p], [mouse]))

    # no_activity_when_landmarks_far_from_object
    p = person()
    p.wrists = [[0.2, 0.2]]
    kb = Obj(10001, "keyboard", (0.8, 0.8, 0.9, 0.85))
    assert infer_activities([p], [kb]) == []

    # no_landmarks_means_no_object_activity
    p = person()
    mouse = Obj(10001, "mouse", (0.35, 0.3, 0.45, 0.4))
    assert infer_activities([p], [mouse]) == []

    # furniture_is_context_not_activity
    p = person()
    p.wrists = [[0.40, 0.60]]
    chair = Obj(10001, "chair", (0.35, 0.55, 0.45, 0.65))
    assert infer_activities([p], [chair]) == []

    # people_together_when_near
    a, b = person(10), person(11)
    acts = infer_activities([a, b], [])
    assert any(x.label == "people_together" and x.participants == [10, 11] for x in acts)

    # seated_at_workstation: upper+upright+stationary adjacent to a workstation class
    p = person(10)
    p.body_coverage, p.torso_attitude, p.motion_state = "upper", "upright", "stationary"
    kb = Obj(10001, "keyboard", (0.45, 0.65, 0.6, 0.75))  # adjacent within 0.10 margin
    acts = infer_activities([p], [kb])
    assert any(x.label == "seated_at_workstation" and x.participants == [10, 10001]
               for x in acts)
    p.motion_state = "moving"   # locomoting → no claim
    assert not any(x.label == "seated_at_workstation"
                   for x in infer_activities([p], [kb]))

    # partially_in_frame: partial coverage + bbox clipped by the frame edge
    p = person(10)
    p.body_coverage = "upper"
    p.bbox = (0.0, 0.2, 0.2, 0.7)
    acts = infer_activities([p], [])
    assert any(x.label == "partially_in_frame" and x.participants == [10] for x in acts)


def _check_fusion_step():
    # repeated_raw_debounces_then_journals_an_event (fusion.rs mesh wiring)
    def typing_world(f):
        p = Person(10)
        p.wrists = [[0.40, 0.58]]
        kb = Obj(10001, "keyboard", (0.35, 0.55, 0.45, 0.62))
        return [p], [kb]

    step = FusionStep()
    for f in range(2):
        people, objects = typing_world(f)
        acts, _, _ = step.tick(people, objects, 0.1 * f, f)
        assert acts == [], f"frame {f}: still a candidate"
    people, objects = typing_world(2)
    acts, _, _ = step.tick(people, objects, 0.2, 2)
    assert len(acts) == 1 and acts[0].label == "typing"
    f = 3
    while f < 33:
        people, objects = typing_world(f)
        step.tick(people, objects, 0.1 * f, f)
        f += 1
    last_events = []
    for g in range(ACTIVITY_EXIT_DWELL):
        _, ev, _ = step.tick([], [], 0.1 * (f + g), f + g)
        if ev:
            last_events = ev
    assert len(last_events) == 1 and last_events[0].label == "typing"
    assert last_events[0].duration_s > 2.0

    # stale_frame_id_does_not_advance_dwell
    step = FusionStep()
    for _ in range(10):
        people, objects = typing_world(5)
        acts, _, _ = step.tick(people, objects, 0.0, 5)
        assert acts == [], "same frame repeated must not confirm"


def _selfcheck():
    _check_fsm()
    _check_anomaly()
    _check_infer()
    _check_fusion_step()
    print("selfcheck OK: FSM (18 scenarios), anomaly (6), infer_activities (13), "
          "frame-gated FusionStep (2) — all match the Rust unit tests")


if __name__ == "__main__":
    _selfcheck()

In [ ]:
#!/usr/bin/env python3
"""Faithful Python port of MVM PersonTracker association (mvm-core: tracker.rs + kalman.rs).

Phase 15.3 lab bench. Same choice as LJ2's stabilizer port: run the production logic in
Python so we can sweep hyperparameters and bootstrap CIs cheaply. The Rust port of whatever
wins (plus a bit-exact golden test Python<->Rust) is deferred to 15.6 — there we also measure
the real Rust code on the live camera.

Ported byte-for-byte from the Rust (constants quoted at their definitions):
  - KalmanBBox: 4 independent scalar constant-velocity filters [pos, vel], F=[[1,1],[0,1]],
    H=[1,0]. Q_POS=1.0 Q_VEL=0.01 R_POS=1.0 R_SIZE=10.0 P_POS_INIT=10.0 P_VEL_INIT=100.0.
  - associate_core: predict-all -> missed+=1 -> Hungarian stage-1 (cost = -(iou + W_OCM*mom))
    gated by IOU_GATE=0.3 -> re-link cascade (cosine / label / center-distance) -> spawn ->
    ByteTrack stage-2 (low boxes) -> inertia advance -> expire at timeout.
  - lifecycle FSM next_lifecycle (reported only; with no masks it does not alter ids).

The tunable knobs the 15.3 experiments move are EXPOSED as tracker attributes (defaults =
production): iou_gate, q_pos, q_vel, r_pos, relink_center_dist, obs_vel_ema, and one structural
flag `center_relink_lowconf_id` (hypothesis A: let the geometric center re-link fire even when a
stale low-confidence identity is present, instead of skipping it).

Mask-IoU term (W_MASK) is inert here (no silhouettes in the bench) exactly as in prod on the
<2-person / no-seg path, so it is omitted. Note W_MASK is zero regardless in the mask-less path,
so `mask_cue_on` has no effect on id assignment — verified against the Rust gate.
"""
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
from scipy.optimize import linear_sum_assignment

# ── production constants (tracker.rs) ──────────────────────────────────────────
IOU_GATE = 0.3          # ln 69
W_OCM = 0.05            # ln 73
OBS_VEL_EMA = 0.5       # ln 83
DIR_EPS = 1e-4          # ln 85
CONFIRM_AGE = 3         # ln 104
LOST_AFTER_MISSED = 30  # ln 107
# kalman.rs
Q_POS, Q_VEL = 1.0, 0.01
R_POS, R_SIZE = 1.0, 10.0
P_POS_INIT, P_VEL_INIT = 10.0, 100.0
# config.rs defaults (re-link)
RELINK_WINDOW = 90
RELINK_MIN_CONF = 0.5
RELINK_COS = 0.5
RELINK_CENTER_DIST = 0.15
TIMEOUT_FRAMES = 1800


def iou(a, b):
    """a,b = [x1,y1,x2,y2]. Matches BBox::iou."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def ocm_consistency(vel, delta):
    vlen = (vel[0] ** 2 + vel[1] ** 2) ** 0.5
    dlen = (delta[0] ** 2 + delta[1] ** 2) ** 0.5
    if vlen < DIR_EPS or dlen < DIR_EPS:
        return 0.0
    return (vel[0] * delta[0] + vel[1] * delta[1]) / (vlen * dlen)


class ScalarKF:
    __slots__ = ("pos", "vel", "p_pp", "p_pv", "p_vv", "q_pos", "q_vel", "r")

    def __init__(self, pos, q_pos, q_vel, r):
        self.pos = pos; self.vel = 0.0
        self.p_pp = P_POS_INIT; self.p_pv = 0.0; self.p_vv = P_VEL_INIT
        self.q_pos = q_pos; self.q_vel = q_vel; self.r = r

    def predict(self):
        self.pos += self.vel
        p_pp = self.p_pp + 2.0 * self.p_pv + self.p_vv + self.q_pos
        p_pv = self.p_pv + self.p_vv
        p_vv = self.p_vv + self.q_vel
        self.p_pp, self.p_pv, self.p_vv = p_pp, p_pv, p_vv
        return self.pos

    def update(self, observed):
        s = self.p_pp + self.r
        k_pos = self.p_pp / s
        k_vel = self.p_pv / s
        y = observed - self.pos
        self.pos += k_pos * y
        self.vel += k_vel * y
        p_pp = (1.0 - k_pos) * self.p_pp
        p_pv = (1.0 - k_pos) * self.p_pv
        p_vv = -k_vel * self.p_pv + self.p_vv
        self.p_pp, self.p_pv, self.p_vv = p_pp, p_pv, p_vv


def to_cwh(b):
    return ((b[0] + b[2]) * 0.5, (b[1] + b[3]) * 0.5, b[2] - b[0], b[3] - b[1])


def from_cwh(cx, cy, w, h):
    return [cx - w * 0.5, cy - h * 0.5, cx + w * 0.5, cy + h * 0.5]


class KalmanBBox:
    def __init__(self, bbox, q_pos, q_vel, r_pos, r_size):
        cx, cy, w, h = to_cwh(bbox)
        self.cx = ScalarKF(cx, q_pos, q_vel, r_pos)
        self.cy = ScalarKF(cy, q_pos, q_vel, r_pos)
        self.w = ScalarKF(w, q_pos, q_vel, r_size)
        self.h = ScalarKF(h, q_pos, q_vel, r_size)

    def predict(self):
        return from_cwh(self.cx.predict(), self.cy.predict(),
                        max(0.0, self.w.predict()), max(0.0, self.h.predict()))

    def update(self, observed):
        cx, cy, w, h = to_cwh(observed)
        self.cx.update(cx); self.cy.update(cy); self.w.update(w); self.h.update(h)

    def smoothed_bbox(self):
        return from_cwh(self.cx.pos, self.cy.pos, max(0.0, self.w.pos), max(0.0, self.h.pos))


@dataclass
class Detection:
    bbox: list
    identity: Optional[str] = None
    identity_confidence: Optional[float] = None
    embedding: Optional[np.ndarray] = None


class Track:
    __slots__ = ("id", "age", "missed", "lifecycle", "kalman", "obs_velocity",
                 "last_obs_center", "identity", "bbox")

    def __init__(self, tid, bbox, kf):
        self.id = tid; self.age = 1; self.missed = 0
        self.lifecycle = "Tentative"; self.kalman = kf
        self.obs_velocity = None; self.last_obs_center = None
        self.identity = None; self.bbox = list(bbox)


def next_lifecycle(prev, age, missed, locked):
    if missed == 0:
        if age < CONFIRM_AGE:
            return "Tentative"
        return "IdLocked" if locked else "Confirmed"
    if prev in ("Tentative", "Lost"):
        return "Lost"
    if missed >= LOST_AFTER_MISSED:
        return "Lost"
    return "Occluded"


def _solve(cost):
    """Kuhn-Munkres minimising total cost; INF = forbidden, never chosen (a row with only
    forbidden options -> None). Mirrors mvm-core::assignment::solve via scipy + big-sentinel."""
    if not cost or not cost[0]:
        return [None] * len(cost)
    C = np.array(cost, float)
    finite = C[np.isfinite(C)]
    big = (finite.max() + 1.0) * (C.shape[0] * C.shape[1] + 1) if finite.size else 1.0
    Cf = np.where(np.isfinite(C), C, big)
    rows, cols = linear_sum_assignment(Cf)
    out = [None] * C.shape[0]
    for r, c in zip(rows, cols):
        if np.isfinite(C[r, c]):        # forbidden pairs (were INF) are dropped
            out[r] = int(c)
    return out


class PersonTracker:
    """Port of PersonTracker. Tunable knobs default to production values."""

    def __init__(self, iou_gate=IOU_GATE, q_pos=Q_POS, q_vel=Q_VEL, r_pos=R_POS, r_size=R_SIZE,
                 relink_center_dist=RELINK_CENTER_DIST, obs_vel_ema=OBS_VEL_EMA,
                 relink_window=RELINK_WINDOW, relink_min_conf=RELINK_MIN_CONF,
                 relink_cos=RELINK_COS, timeout=TIMEOUT_FRAMES,
                 center_relink_lowconf_id=False):
        self.tracks = {}
        self.next_id = 10          # ephemeral ids start at 10 (0-9 reserved for gallery)
        self.free_ids = []
        self.iou_gate = iou_gate
        self.q_pos, self.q_vel, self.r_pos, self.r_size = q_pos, q_vel, r_pos, r_size
        self.relink_center_dist = relink_center_dist
        self.obs_vel_ema = obs_vel_ema
        self.relink_window = relink_window
        self.relink_min_conf = relink_min_conf
        self.relink_cos = relink_cos
        self.timeout = timeout
        self.center_relink_lowconf_id = center_relink_lowconf_id  # hypothesis-A structural fix

    def _alloc_id(self):
        if self.free_ids:
            return self.free_ids.pop(0)
        i = self.next_id; self.next_id += 1
        return i

    def _record_obs(self, t, obs_c):
        if t.last_obs_center is not None:
            gap = max(1, t.missed)
            inst = [(obs_c[0] - t.last_obs_center[0]) / gap,
                    (obs_c[1] - t.last_obs_center[1]) / gap]
            if t.obs_velocity is None:
                t.obs_velocity = inst
            else:
                e = self.obs_vel_ema
                t.obs_velocity = [e * inst[0] + (1 - e) * t.obs_velocity[0],
                                  e * inst[1] + (1 - e) * t.obs_velocity[1]]
        t.last_obs_center = list(obs_c)

    def tick(self, detections, low=()):
        """One frame. detections = list[Detection]; low = list[bbox]. Returns id per detection."""
        predicted = {tid: t.kalman.predict() for tid, t in self.tracks.items()}
        for t in self.tracks.values():
            t.missed += 1

        matched_ids = []
        ids = []
        cand_ids = sorted(predicted.keys())

        # ── stage-1 Hungarian (iou + W_OCM*momentum), gated by iou_gate ──
        if not detections or not cand_ids:
            primary = [None] * len(detections)
        else:
            cost = []
            for det in detections:
                dcx = (det.bbox[0] + det.bbox[2]) * 0.5
                dcy = (det.bbox[1] + det.bbox[3]) * 0.5
                row = []
                for tid in cand_ids:
                    ov = iou(det.bbox, predicted[tid])
                    if ov <= self.iou_gate:
                        row.append(float("inf")); continue
                    t = self.tracks[tid]
                    if t.obs_velocity is not None and t.last_obs_center is not None:
                        mom = ocm_consistency(t.obs_velocity,
                                              [dcx - t.last_obs_center[0], dcy - t.last_obs_center[1]])
                    else:
                        mom = 0.0
                    row.append(-(ov + W_OCM * mom))
                cost.append(row)
            primary = [cand_ids[j] if j is not None else None for j in _solve(cost)]

        for pid in primary:
            if pid is not None:
                matched_ids.append(pid)

        # ── per-detection: primary, else re-link cascade, else spawn ──
        for d, det in enumerate(detections):
            best_id = primary[d]

            # tier 1: cosine SFace embedding
            if best_id is None and det.embedding is not None:
                best_cos = self.relink_cos
                for tid, t in self.tracks.items():
                    if tid in matched_ids: continue
                    if t.missed == 0 or t.missed > self.relink_window: continue
                    # (track embeddings not modeled in the bench)
                    _ = t
            # tier 2: label-equality re-link (no embedding)
            if best_id is None and det.embedding is None:
                if det.identity is not None and det.identity_confidence is not None \
                        and det.identity_confidence >= self.relink_min_conf:
                    best_missed = None
                    for tid, t in self.tracks.items():
                        if tid in matched_ids: continue
                        if t.missed == 0 or t.missed > self.relink_window: continue
                        if t.identity == det.identity and (best_missed is None or t.missed < best_missed):
                            best_missed = t.missed; best_id = tid

            # tier 3: center-distance re-link (anonymous geometric fallback, OCR virtual pos)
            # PROD gate: det has no embedding AND no identity.
            # Hypothesis-A fix: also allow when identity is present but LOW-confidence
            # (a stale intermittent face) — the geometric match should not be blocked by it.
            det_id_blocks = det.identity is not None
            if self.center_relink_lowconf_id:
                det_id_blocks = det.identity is not None and \
                    (det.identity_confidence is None or det.identity_confidence >= self.relink_min_conf)
            if best_id is None and det.embedding is None and not det_id_blocks:
                dcx = (det.bbox[0] + det.bbox[2]) * 0.5
                dcy = (det.bbox[1] + det.bbox[3]) * 0.5
                best_dist = self.relink_center_dist
                # names actively claimed by a CONFIDENT id on some detection THIS frame —
                # never geometrically steal a track whose person is being identified elsewhere.
                claimed = {dd.identity for dd in detections
                           if dd.identity is not None and dd.identity_confidence is not None
                           and dd.identity_confidence >= self.relink_min_conf}
                for tid, t in self.tracks.items():
                    if tid in matched_ids: continue
                    if t.missed == 0 or t.missed > self.relink_window: continue
                    # PROD: skip any identified track. FIX (hypothesis A): a body-primary walker
                    # gets ONE confident face early -> track.identity sticks -> this skip
                    # permanently disables the anonymous geometric fallback for it -> every
                    # IoU gap spawns a phantom. With the fix, an identified track is still
                    # eligible for a geometric re-link UNLESS its name is being actively
                    # claimed by a confident detection elsewhere this frame.
                    if t.identity is not None:
                        if not self.center_relink_lowconf_id or t.identity in claimed:
                            continue
                    if t.last_obs_center is not None and t.obs_velocity is not None:
                        gap = float(t.missed)
                        vcx = t.last_obs_center[0] + t.obs_velocity[0] * gap
                        vcy = t.last_obs_center[1] + t.obs_velocity[1] * gap
                    elif t.last_obs_center is not None:
                        vcx, vcy = t.last_obs_center
                    else:
                        vcx = (t.bbox[0] + t.bbox[2]) * 0.5; vcy = (t.bbox[1] + t.bbox[3]) * 0.5
                    dd = ((dcx - vcx) ** 2 + (dcy - vcy) ** 2) ** 0.5
                    if dd < best_dist:
                        best_dist = dd; best_id = tid

            if best_id is not None:
                t = self.tracks[best_id]
                t.kalman.update(det.bbox)
                t.bbox = t.kalman.smoothed_bbox()
                t.age += 1
                self._record_obs(t, [(det.bbox[0] + det.bbox[2]) * 0.5,
                                     (det.bbox[1] + det.bbox[3]) * 0.5])
                t.missed = 0
                if det.identity is not None and det.identity_confidence is not None \
                        and det.identity_confidence >= self.relink_min_conf:
                    t.identity = det.identity     # only a confident id sticks to the track
                if best_id not in matched_ids:
                    matched_ids.append(best_id)
                the_id = best_id
            else:
                the_id = self._alloc_id()
                kf = KalmanBBox(det.bbox, self.q_pos, self.q_vel, self.r_pos, self.r_size)
                nt = Track(the_id, det.bbox, kf)
                # seed the first observation anchor (apply_detection does this in prod)
                nt.last_obs_center = [(det.bbox[0] + det.bbox[2]) * 0.5,
                                      (det.bbox[1] + det.bbox[3]) * 0.5]
                if det.identity is not None and det.identity_confidence is not None \
                        and det.identity_confidence >= self.relink_min_conf:
                    nt.identity = det.identity
                self.tracks[the_id] = nt
            ids.append(the_id)

        # ── stage-2 ByteTrack low-conf recovery ──
        for lb in low:
            lcx = (lb[0] + lb[2]) * 0.5; lcy = (lb[1] + lb[3]) * 0.5
            best_score = -float("inf"); best_id = None
            for tid, t in self.tracks.items():
                if t.missed == 0 or tid in matched_ids: continue
                if tid not in predicted: continue
                ov = iou(lb, predicted[tid])
                if ov <= self.iou_gate: continue
                if t.obs_velocity is not None and t.last_obs_center is not None:
                    mom = ocm_consistency(t.obs_velocity, [lcx - t.last_obs_center[0], lcy - t.last_obs_center[1]])
                else:
                    mom = 0.0
                score = ov + W_OCM * mom
                if score > best_score:
                    best_score = score; best_id = tid
            if best_id is not None:
                t = self.tracks[best_id]
                t.kalman.update(lb); t.bbox = t.kalman.smoothed_bbox(); t.age += 1
                self._record_obs(t, [lcx, lcy]); t.missed = 0
                matched_ids.append(best_id)

        # ── inertia advance + lifecycle + expire ──
        for tid, t in self.tracks.items():
            if tid not in matched_ids and tid in predicted:
                t.bbox = predicted[tid]
            t.lifecycle = next_lifecycle(t.lifecycle, t.age, t.missed, locked=False)

        expired = [tid for tid, t in self.tracks.items() if t.missed >= self.timeout]
        for tid in expired:
            del self.tracks[tid]
            self.free_ids.append(tid); self.free_ids.sort()

        return ids


# ── self-check against known Rust unit-test scenarios ──────────────────────────
def _selfcheck():
    # 1) moving object stays one track (kalman.rs tracker_uses_predicted_bbox_for_iou)
    tk = PersonTracker()
    for i in range(5):
        x1 = 0.10 + i * 0.03
        tk.tick([Detection([x1, 0.3, x1 + 0.20, 0.7])])
    assert len(tk.tracks) == 1, f"moving object fragmented: {len(tk.tracks)} tracks"

    # 2) idle object -> exactly one stable id over many frames
    tk = PersonTracker()
    last = None
    for _ in range(50):
        ids = tk.tick([Detection([0.4, 0.2, 0.6, 0.9])])
        last = ids[0]
    assert len(tk.tracks) == 1 and last == 10, f"idle churned: {len(tk.tracks)} tracks id={last}"

    # 3) occlusion gap then re-appear nearby -> re-link (no new id), anonymous center tier
    tk = PersonTracker()
    for i in range(6):                       # establish + build obs velocity moving right
        x1 = 0.10 + i * 0.03
        tk.tick([Detection([x1, 0.3, x1 + 0.20, 0.7])])
    tk.tick([])                              # 1 missed frame
    ids = tk.tick([Detection([0.10 + 7 * 0.03, 0.3, 0.10 + 7 * 0.03 + 0.20, 0.7])])
    assert ids[0] == 10, f"occluded moving body churned a new id: {ids[0]}"
    print("selfcheck OK: moving=1 track, idle=1 stable id, occlusion re-links")


if __name__ == "__main__":
    _selfcheck()

### 3.2 The bench: emitter, full-stack runner, event scoring

Also the lab source (`lj4_bench.py`), embedded with its repo-local imports stripped (everything
already lives in this namespace). Key honesty details:

- **FLOOR** = clean injection; **STRESS** = the LJ3 live-detector model (bursty occlusion,
  bbox jitter, intermittent early face — the dead-zone trigger) plus per-frame posture flips.
- Every noise source draws from its **own** seeded generator: switching one off in an ablation
  must not reshuffle the others (the first run caught exactly this — an unpaired "no posture
  flip" ablation scored *worse* than baseline).
- Scoring is event-level: label + participants (mapped through tracker-committed ids) + start/end
  within tolerance. Unmatched emitted events split into **fragments** (right activity, broken
  episode) vs **foreign** (invented activity) — the attribution hinge.

In [ ]:
#!/usr/bin/env python3
"""LJ4 bench (S1/S2): drive golden scenarios through the FULL ported temporal stack —
tracker (lj3_tracker) -> posture stabilizer (LJ2 port, online) -> infer_activities ->
Activity FSM -> anomaly (lj4_fusion) — and score journal events against the scenario's
known-by-construction truth.

Frozen bench-protocol component "Simulator/FSM": leading metric = EVENT precision/recall
vs the scenario (label + participants + start/stop within tolerance), plus anomaly
false-alarm rate on quiet scenarios and anomaly hit on falls. Grouped bootstrap CI over
scenario instances (N=1000, seed=0).

Emitters (the LJ3 protocol + one posture knob; comparable only within one protocol):
  FLOOR  — clean injection: exact bbox, true kinematic posture, no identity.
  STRESS — live-like detector defects: bursty occlusion dropout, bbox jitter,
           intermittent early face identity (the 15.3 dead-zone trigger), and
           per-frame posture misclassification (flip to an adjacent class).

Layer ablations at STRESS med attribute the degradation:
  tracker=fix15.3 (stale-identity center re-link), stabilizer=hybrid (LJ2 sl_dwell
  veto), face=0 (no identity stamping), flip=0 (clean posture).

S2 rides the same runs: prod AnomalyScorer (standing->lying <= 3 s window, NO velocity)
vs the LJ2 E2 physics rule (reached floor AND peak downward pelvis velocity) on G3/G4.

Run:  ../venv/bin/python3 spike_pose/lj4_bench.py
"""
import numpy as np


FPS = 15
SEED = 0
N_BOOT = 1000
POSTURE_CLASSES = ["standing", "sitting", "lying"]
POSTURE_CONF_DEFAULT = 0.9      # injected classifier confidence (scenario meta)

# STRESS protocols: (dropout, jitter, face_frames, burst, p_flip). dropout/jitter/face/
# burst are the LJ3 values (same live-detector model); p_flip adds posture noise.
STRESS = {"mild":  (0.06, 0.010, 3, (2, 4), 0.03),
          "med":   (0.10, 0.020, 3, (2, 4), 0.06),
          "heavy": (0.15, 0.030, 3, (2, 5), 0.10)}
FLOOR = (0.0, 0.0, 0, (2, 4), 0.0)

# LJ2 E2 fall rule (results/metrics.json["lj2"]): reached floor AND fast descent.
E2_FLOOR_M = 0.35
E2_VEL_MS = 1.13        # calibrated at 60 fps (LJ2)
E2_VEL_RECAL = 1.05     # 15 fps recalibration candidate: the production cadence smears
                        # the peak (G4_85_15 reads 1.12 < 1.13 while the worst quiet
                        # scenario reads 0.99) — measured here, decided by the table
E2_WIN_S = 0.4


# ── online posture stabilizer (LJ2 port of posture.rs, stateful) ───────────────

class Stabilizer:
    """Per-track online form of lj2_stabilizer.run_dwell / run_hybrid. sl_dwell=None ->
    production dwell machine; sl_dwell=12 -> the 15.2 shipped hybrid (SL veto)."""

    def __init__(self, dwell=3, min_conf=0.5, sl_dwell=None):
        self.dwell, self.min_conf, self.sl_dwell = dwell, min_conf, sl_dwell
        self.committed = None
        self.cand = None
        self.streak = 0

    def step(self, lab, conf):
        need = self.dwell
        if (self.sl_dwell is not None and self.committed is not None
                and {self.committed, lab} == {"standing", "lying"}):
            need = self.sl_dwell
        if conf < self.min_conf:
            self.cand = self.committed
            self.streak = 0
        elif self.committed is None:
            self.committed = lab
            self.cand = lab
            self.streak = 0
        elif lab == self.committed:
            self.cand = lab
            self.streak = 0
        else:
            if self.cand == lab:
                self.streak += 1
            else:
                self.cand = lab
                self.streak = 1
            if self.streak >= need:
                self.committed = lab
                self.streak = 0
        return self.committed if self.committed is not None else lab


# ── data access ────────────────────────────────────────────────────────────────

def load():
    d = np.load(GOLD, allow_pickle=True)
    meta = json.loads(str(d["meta_json"]))
    return d, meta


def frames_of(d, inst, n_frames):
    """frame idx -> list of truth-row dicts for this instance."""
    s = d["inst"] == inst
    by = collections.defaultdict(list)
    for i in np.where(s)[0]:
        by[int(d["frame"][i])].append(dict(
            person=int(d["person"][i]), bbox=d["bbox"][i].astype(float),
            vis=bool(d["vis"][i]), posture=str(d["posture"][i]),
            wrists=d["wrists"][i].astype(float), pelvis3d=d["pelvis3d"][i].astype(float),
            pos3d=tuple(float(v) for v in d["pos3d"][i])))
    return [by.get(f, []) for f in range(n_frames)]


def objects_of(d, inst):
    s = d["obj_inst"] == inst
    return [Obj(int(d["obj_id"][i]), str(d["obj_cls"][i]), tuple(d["obj_bbox"][i]))
            for i in np.where(s)[0]]


# ── emitter (LJ3 semantics + posture flip) ─────────────────────────────────────

def emit(frames, seed_key, dropout=0.0, jitter=0.0, face_frames=0, burst=(2, 4), p_flip=0.0):
    """-> per-frame list of (Detection, truth_row). Bursty occlusion (live misses come
    in runs), bbox jitter, confident face identity on a person's first sightings (the
    dead-zone trigger), posture flipped to another class with p_flip.

    Each noise source draws from its OWN generator (seeded by (seed, instance, source)):
    turning one source off in an ablation must not shift the other sources' streams —
    otherwise 'no posture flip' silently reshuffles the dropout pattern and the ablation
    compares two different worlds (caught on the first run: no-flip scored WORSE)."""
    seed, inst_idx = seed_key
    r_drop, r_jit, r_flip = (np.random.default_rng([seed, inst_idx, salt])
                             for salt in (0, 1, 2))
    seen = collections.Counter()
    occ = collections.defaultdict(int)
    out = []
    for fl in frames:
        cur = []
        for row in fl:
            if not row["vis"]:
                continue
            p = row["person"]
            if occ[p] > 0:
                occ[p] -= 1
                continue
            if dropout and r_drop.random() < dropout:
                occ[p] = int(r_drop.integers(burst[0], burst[1] + 1)) - 1
                continue
            b = row["bbox"].copy()
            if jitter:
                b[:2] += r_jit.normal(0, jitter, 2)
                b[2:] += r_jit.normal(0, jitter, 2)
            ident = conf = None
            seen[p] += 1
            if face_frames and seen[p] <= face_frames:
                ident, conf = f"P{p}", 0.9
            lab = row["posture"]
            if p_flip and r_flip.random() < p_flip:
                lab = r_flip.choice([c for c in POSTURE_CLASSES if c != lab])
            cur.append((Detection(list(b), identity=ident, identity_confidence=conf),
                        {**row, "posture_obs": lab}))
        out.append(cur)
    return out


# ── the full-stack runner ──────────────────────────────────────────────────────

def run_instance(emitted, objects, n_frames, tracker_kw=None, stab_kw=None,
                 journal_kw=None):
    """Drive tracker -> stabilizer -> fusion over one instance. Returns dict with
    journal events (open episodes closed at the end and tagged), anomaly trace,
    tid->person mapping, per-person committed ids, and per-frame pelvis3d truth."""
    tracker = PersonTracker(**(tracker_kw or {}))
    step = FusionStep(journal_kw=journal_kw)
    stabs = {}
    tid_person = collections.Counter()
    ids_of = collections.defaultdict(list)
    events, anomaly = [], []
    for f in range(n_frames):
        dets = [dt for dt, _ in emitted[f]]
        rows = [r for _, r in emitted[f]]
        ids = tracker.tick(dets)
        people = []
        for tid, row in zip(ids, rows):
            tid_person[(tid, row["person"])] += 1
            ids_of[row["person"]].append(tid)
            st = stabs.setdefault(tid, Stabilizer(**(stab_kw or {})))
            lab = st.step(row["posture_obs"], POSTURE_CONF_DEFAULT)
            people.append(Person(
                track_id=tid, bbox=tuple(row["bbox"]), position_3d=row["pos3d"],
                posture=(lab, POSTURE_CONF_DEFAULT),
                wrists=[list(w) for w in row["wrists"]]))
        _, ev, an = step.tick(people, objects, f / FPS, f)
        events.extend([(e, False) for e in ev])
        anomaly.append(an)
    for inst_a in step.journal.active():        # open episodes at scenario end
        events.append((_event_from(inst_a), True))
    tid2p = {}
    for (tid, p), n in tid_person.items():
        if tid not in tid2p or n > tid_person[(tid, tid2p[tid])]:
            tid2p[tid] = p
    return dict(events=events, anomaly=np.array(anomaly), tid2p=tid2p, ids_of=ids_of)


# ── scoring vs expected ────────────────────────────────────────────────────────

def score_instance(res, meta_inst):
    """-> dict(tp, fp, fn, fa (quiet violated), fire_ok (fall fired / None),
    excess, swaps)."""
    fps = meta_inst["fps"]
    t_end = meta_inst["n_frames"] / fps
    tid2p = res["tid2p"]

    def mapped(ev):
        persons = frozenset(tid2p.get(pid, -1) for pid in ev.participants if pid < 10000)
        objs = frozenset(pid for pid in ev.participants if pid >= 10000)
        return persons, objs

    emitted = [(ev, open_, *mapped(ev)) for ev, open_ in res["events"]]
    used = [False] * len(emitted)
    tp = fn = 0
    for exp in meta_inst["expected"]:
        if exp["kind"] != "activity":
            continue
        want_p = frozenset(v for v in (exp.get("person"), exp.get("person2")) if v is not None)
        want_o = frozenset([exp["obj"]] if "obj" in exp else [])
        got = False
        for i, (ev, open_, ps, os_) in enumerate(emitted):
            if used[i] or ev.label != exp["label"] or ps != want_p or os_ != want_o:
                continue
            end = t_end if open_ else ev.ended_at
            if (abs(ev.started_at - exp["t_start"]) <= exp["tol_start"]
                    and abs(end - exp["t_end"]) <= exp["tol_end"]):
                used[i] = True
                got = True
                break
        tp += got
        fn += not got
    # FP breakdown: same key as SOME expected but wrong timing / a split episode
    # ("fragment") vs an event no expected mentions at all ("foreign").
    fp = fp_frag = fp_foreign = 0
    exp_keys = {(e["label"],
                 frozenset(v for v in (e.get("person"), e.get("person2")) if v is not None),
                 frozenset([e["obj"]] if "obj" in e else []))
                for e in meta_inst["expected"] if e["kind"] == "activity"}
    for i, (ev, open_, ps, os_) in enumerate(emitted):
        if used[i]:
            continue
        fp += 1
        if (ev.label, ps, os_) in exp_keys:
            fp_frag += 1
        else:
            fp_foreign += 1

    fa = fire_ok = None
    an = res["anomaly"]
    for exp in meta_inst["expected"]:
        if exp["kind"] == "anomaly_quiet":
            fa = bool((an >= exp["max"]).any())
        elif exp["kind"] == "anomaly_fire":
            a = int(max(0, exp["t_from"] * fps))
            b = int(min(len(an), exp["t_to"] * fps + 1))
            fire_ok = bool((an[a:b] >= exp["min"]).any())

    excess = sum(max(0, len(set(v)) - 1) for v in res["ids_of"].values())
    id2ps = collections.defaultdict(set)
    for p, ids in res["ids_of"].items():
        for i in ids:
            id2ps[i].add(p)
    swaps = sum(1 for ps in id2ps.values() if len(ps) > 1)
    minutes = meta_inst["n_frames"] / fps / 60.0
    return dict(tp=tp, fp=fp, fn=fn, fp_frag=fp_frag, fp_foreign=fp_foreign,
                fa=fa, fire_ok=fire_ok, excess=excess, swaps=swaps, minutes=minutes)


# ── evaluation over all instances ──────────────────────────────────────────────

def evaluate(d, meta, proto, tracker_kw=None, stab_kw=None, journal_kw=None,
             seed=SEED, groups=None):
    """proto = (dropout, jitter, face, burst, p_flip). Fresh RNG per call: paired
    configs with equal emitter params see identical detection streams."""
    dropout, jitter, face, burst, p_flip = proto
    rows = {}
    for idx, (inst, m) in enumerate(meta.items()):
        if groups and m["group"] not in groups:
            continue
        frames = frames_of(d, inst, m["n_frames"])
        emitted = emit(frames, (seed, idx), dropout, jitter, face, burst, p_flip)
        res = run_instance(emitted, objects_of(d, inst), m["n_frames"],
                           tracker_kw, stab_kw, journal_kw)
        rows[inst] = (score_instance(res, m), res)
    return rows


def summarize(rows):
    s = collections.Counter()
    quiet_n = quiet_bad = fire_n = fire_hit = 0
    for sc, _ in rows.values():
        s["tp"] += sc["tp"]; s["fp"] += sc["fp"]; s["fn"] += sc["fn"]
        s["fp_frag"] += sc["fp_frag"]; s["fp_foreign"] += sc["fp_foreign"]
        s["excess"] += sc["excess"]; s["swaps"] += sc["swaps"]
        s["minutes"] += sc["minutes"]
        if sc["fa"] is not None:
            quiet_n += 1; quiet_bad += sc["fa"]
        if sc["fire_ok"] is not None:
            fire_n += 1; fire_hit += sc["fire_ok"]
    prec = s["tp"] / (s["tp"] + s["fp"]) if s["tp"] + s["fp"] else 1.0
    rec = s["tp"] / (s["tp"] + s["fn"]) if s["tp"] + s["fn"] else 1.0
    return dict(precision=prec, recall=rec, tp=s["tp"], fp=s["fp"], fn=s["fn"],
                fp_frag=s["fp_frag"], fp_foreign=s["fp_foreign"],
                fa_rate=quiet_bad / quiet_n if quiet_n else 0.0, quiet_n=quiet_n,
                fire_rate=fire_hit / fire_n if fire_n else None, fire_n=fire_n,
                excess_per_min=s["excess"] / s["minutes"], swaps=s["swaps"])


def ci_prf(rows, n_boot=N_BOOT, seed=SEED):
    """Grouped bootstrap over instances: precision & recall CIs."""
    arr = np.array([(sc["tp"], sc["fp"], sc["fn"]) for sc, _ in rows.values()], float)
    rng = np.random.default_rng(seed)
    ps, rs = [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(arr), len(arr))
        tp, fp, fn = arr[i].sum(0)
        ps.append(tp / (tp + fp) if tp + fp else 1.0)
        rs.append(tp / (tp + fn) if tp + fn else 1.0)
    pct = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return pct(ps), pct(rs)


# ── S2: prod anomaly vs the LJ2 E2 rule on the same runs ───────────────────────

def e2_stats(frames, fps=FPS):
    """LJ2 E2 on the injected world pelvis: reached floor (min pelvis y <= E2_FLOOR_M)
    AND peak windowed descent speed >= E2_VEL_MS. Median-filtered heights, 0.4 s drop
    window — the lj2_fall_velocity robust recipe at the production cadence."""
    y = np.array([fl[0]["pelvis3d"][1] if fl else np.nan for fl in frames], float)
    y = y[~np.isnan(y)]
    if len(y) < 3:
        return False, np.inf, 0.0
    k = 3
    med = np.array([np.median(y[max(0, i - k // 2):i + k // 2 + 1]) for i in range(len(y))])
    w = max(1, int(round(E2_WIN_S * fps)))
    drops = med[:-w] - med[w:]
    vmax = float(drops.max() / E2_WIN_S) if len(drops) else 0.0
    fired = bool(med.min() <= E2_FLOOR_M and vmax >= E2_VEL_MS)
    return fired, float(med.min()), vmax


def s1b_exit_dwell_sweep(d, meta):
    """The FLOOR run put P/R at 1.0 and the med ablations put ALL precision loss in
    episode fragments that no tracker/stabilizer knob touches -> the suspect is the
    FSM exit grace (5 frames = 0.33 s) vs realistic multi-frame occlusion bursts.
    Sweep it (young raised in step; prod = 5/8)."""
    print("\n" + "-" * 78)
    print("S1b exit-dwell sweep @ STRESS med/heavy — FSM grace vs occlusion bursts")
    print("-" * 78)
    for name, kw in [("prod 5/8", None), ("8/11", dict(exit_dwell=8, exit_young=11)),
                     ("10/13", dict(exit_dwell=10, exit_young=13)),
                     ("12/15", dict(exit_dwell=12, exit_young=15))]:
        for sname in ("med", "heavy"):
            rows = evaluate(d, meta, STRESS[sname], journal_kw=kw)
            s = summarize(rows)
            pci, rci = ci_prf(rows)
            print(f"exit={name:9} {sname:5} {fmt(s, pci, rci)}")


def s1c_g4_diagnostics(d, meta):
    """Why do falls get missed under stress? Trace what the anomaly actually SAW."""
    print("\n" + "-" * 78)
    print("S1c G4 under STRESS med — what reached the anomaly at the critical moment")
    print("-" * 78)
    idx_of = {inst: i for i, inst in enumerate(meta)}
    for inst, m in meta.items():
        if m["group"] != "G4":
            continue
        frames = frames_of(d, inst, m["n_frames"])
        emitted = emit(frames, (SEED, idx_of[inst]), *STRESS["med"])
        res = run_instance(emitted, objects_of(d, inst), m["n_frames"])
        # Reconstruct the STABILIZED posture stream the anomaly consumed.
        seen = []
        stabs = {}
        for f, fl in enumerate(emitted):
            for _, row in fl:
                st = stabs.setdefault(0, Stabilizer())
                seen.append((f, st.step(row["posture_obs"], POSTURE_CONF_DEFAULT)))
        stand = [f for f, l in seen if l == "standing"]
        lying = [f for f, l in seen if l == "lying"]
        gap = (lying[0] - stand[-1]) / FPS if stand and lying and lying[0] > stand[-1] \
            else ("n/a" if not (stand and lying) else "overlap")
        fired = bool((res["anomaly"] >= 0.5).any())
        print(f"{inst:12} fired={str(fired):5} stand-obs={len(stand):3} "
              f"lying-obs={len(lying):3} last-stand->first-lying="
              f"{gap if isinstance(gap, str) else f'{gap:.2f}s'} "
              f"(window {B_FALL_WINDOW}s)")


B_FALL_WINDOW = 3.0


def s2_run(d, meta, inst, proto, idx):
    """Drive tracker+stabilizer over one G3/G4 instance under `proto`, recording what
    a fall rule could actually consume: per-frame delivered (tid, stabilized label,
    observed pelvis y). Also returns the run's prod anomaly trace."""
    m = meta[inst]
    frames = frames_of(d, inst, m["n_frames"])
    emitted = emit(frames, (SEED, idx), *proto)
    res = run_instance(emitted, objects_of(d, inst), m["n_frames"])
    stabs = {}
    stream = []                          # (t, tid, stab_label, pelvis_y)
    tracker = PersonTracker()
    for f, fl in enumerate(emitted):
        ids = tracker.tick([dt for dt, _ in fl])
        for tid, (_, row) in zip(ids, fl):
            st = stabs.setdefault(tid, Stabilizer())
            stream.append((f / FPS, tid, st.step(row["posture_obs"], POSTURE_CONF_DEFAULT),
                           float(row["pelvis3d"][1])))
    return res, stream


def rule_prod_idfree(stream, window=3.0):
    """The prod fall rule with the track-id dependency removed: the standing memory is
    SCENE-level (any body stood recently -> a body lying now is a fall). Motivated by
    the S1c finding: the fall itself churns the id, and the new id 'never stood'."""
    last_stand = None
    for t, _, lab, _ in stream:
        if lab == "standing":
            last_stand = t
        if lab == "lying" and last_stand is not None and t - last_stand <= window:
            return True
    return False


def rule_e2(stream, thresh, fps=FPS):
    """E2 on the DELIVERED pelvis stream (gaps interpolated onto the frame grid — a
    long occlusion smooths the descent and honestly weakens the rule)."""
    obs = [(t, y) for t, _, _, y in stream]
    if len(obs) < 3:
        return False
    t_obs = np.array([t for t, _ in obs])
    y_obs = np.array([y for _, y in obs])
    grid = np.arange(t_obs[0], t_obs[-1], 1.0 / fps)
    y = np.interp(grid, t_obs, y_obs)
    k = 3
    med = np.array([np.median(y[max(0, i - k // 2):i + k // 2 + 1]) for i in range(len(y))])
    w = max(1, int(round(E2_WIN_S * fps)))
    if len(med) <= w:
        return False
    vmax = float((med[:-w] - med[w:]).max() / E2_WIN_S)
    return bool(med.min() <= E2_FLOOR_M and vmax >= thresh)


def s2_table(d, meta):
    print("\n" + "=" * 78)
    print("S2  FALL RULES on G3 (quiet: controlled floor) vs G4 (falls), FLOOR + STRESS med")
    print("    prod        = AnomalyScorer, standing->lying <=3s window, PER-TRACK-ID")
    print("    prod-idfree = same window, scene-level standing memory (churn-robust)")
    print(f"    E2          = floor<={E2_FLOOR_M}m AND descent>={E2_VEL_MS}m/s on delivered pelvis")
    print(f"    E2-recal    = same, threshold {E2_VEL_RECAL} m/s (15 fps recalibration)")
    print("=" * 78)
    idx_of = {inst: i for i, inst in enumerate(meta)}
    for pname, proto in [("FLOOR", FLOOR), ("STRESS med", STRESS["med"])]:
        stats = collections.Counter()
        detail = []
        for inst, m in meta.items():
            if m["group"] not in ("G3", "G4"):
                continue
            g = m["group"]
            res, stream = s2_run(d, meta, inst, proto, idx_of[inst])
            fired = {
                "prod": bool((res["anomaly"] >= 0.5).any()),
                "idfree": rule_prod_idfree(stream),
                "e2": rule_e2(stream, E2_VEL_MS),
                "e2rec": rule_e2(stream, E2_VEL_RECAL),
            }
            want = g == "G4"
            for k, v in fired.items():
                stats[f"{k}_{g}"] += v
            stats[f"{g}_n"] += 1
            miss = [k for k, v in fired.items() if v != want]
            if miss:
                detail.append(f"   {inst}: " + ", ".join(
                    f"{k}={'FALSE ALARM' if fired[k] else 'MISS'}" for k in miss))
        print(f"\n  {pname}:")
        print(f"  {'rule':12} {'G3 false alarms':>16} {'G4 hits':>9}")
        for k, label in [("prod", "prod"), ("idfree", "prod-idfree"),
                         ("e2", "E2 @1.13"), ("e2rec", "E2 @1.05")]:
            print(f"  {label:12} {stats[f'{k}_G3']:>12}/{stats['G3_n']:<3} "
                  f"{stats[f'{k}_G4']:>5}/{stats['G4_n']}")
        for line in detail:
            print(line)


# ── driver ─────────────────────────────────────────────────────────────────────

def fmt(s, pci=None, rci=None):
    p = f"{s['precision']:.3f}" + (f" [{pci[0]:.2f},{pci[1]:.2f}]" if pci else "")
    r = f"{s['recall']:.3f}" + (f" [{rci[0]:.2f},{rci[1]:.2f}]" if rci else "")
    fire = "—" if s["fire_rate"] is None else f"{s['fire_rate']:.2f}"
    return (f"P={p:22} R={r:22} FP(frag/foreign)={s['fp_frag']}/{s['fp_foreign']} "
            f"FA={s['fa_rate']:.2f}({s['quiet_n']}) fall-hit={fire} "
            f"excess/min={s['excess_per_min']:.2f} swaps={s['swaps']}")


def main():
    d, meta = load()
    print(f"instances: {len(meta)}  groups="
          f"{dict(collections.Counter(m['group'] for m in meta.values()))}")

    print("\n" + "=" * 78)
    print("S1  EVENT P/R vs scenario  (leading metric; grouped bootstrap CI over instances)")
    print("=" * 78)
    base = {}
    for name, proto in [("FLOOR", FLOOR)] + [(f"STRESS {k}", v) for k, v in STRESS.items()]:
        rows = evaluate(d, meta, proto)
        s = summarize(rows)
        pci, rci = ci_prf(rows)
        base[name] = rows
        print(f"{name:14} {fmt(s, pci, rci)}")
        if name == "FLOOR":
            for inst, (sc, _) in rows.items():
                if sc["fp"] or sc["fn"] or sc["fa"] or sc["fire_ok"] is False or sc["excess"] or sc["swaps"]:
                    print(f"   !! {inst}: {sc}")

    print("\n" + "-" * 78)
    print("S1 ablations @ STRESS med — attribute the degradation to a layer")
    print("-" * 78)
    med = STRESS["med"]
    for name, kw in [
        ("baseline (prod)", {}),
        ("tracker=fix15.3", dict(tracker_kw=dict(center_relink_lowconf_id=True))),
        ("stab=hybrid sl12", dict(stab_kw=dict(sl_dwell=12))),
        ("no face (no id)", dict(proto=(med[0], med[1], 0, med[3], med[4]))),
        ("no posture flip", dict(proto=(med[0], med[1], med[2], med[3], 0.0))),
    ]:
        proto = kw.pop("proto", med)
        rows = evaluate(d, meta, proto, **kw)
        s = summarize(rows)
        print(f"{name:16} {fmt(s)}")

    s1b_exit_dwell_sweep(d, meta)
    s1c_g4_diagnostics(d, meta)
    s2_table(d, meta)

    print("\npassport:", bench.passport([GOLD]))

## 4. Numbers — S1: the FLOOR gate, then stress with attribution

First the gate: clean injection through the full stack, every instance.

In [ ]:
# imports the embedded bench cell needs at the top of this namespace
import os, sys, json, collections
import numpy as np

print("=" * 78)
print("S1  EVENT P/R vs scenario (grouped bootstrap CI over instances, N=1000, seed=0)")
print("=" * 78)
base_rows = {}
for name, proto in [("FLOOR", FLOOR)] + [(f"STRESS {k}", v) for k, v in STRESS.items()]:
    rows = evaluate(_d, _meta, proto)
    s = summarize(rows)
    pci, rci = ci_prf(rows)
    base_rows[name] = rows
    print(f"{name:14} {fmt(s, pci, rci)}")

print("\nFLOOR flags (findings, not scoring failures):")
for inst, (sc, _) in base_rows["FLOOR"].items():
    if sc["fp"] or sc["fn"] or sc["fa"] or sc["fire_ok"] is False or sc["excess"] or sc["swaps"]:
        print(f"  !! {inst}: fp={sc['fp']} fn={sc['fn']} false_alarm={sc['fa']} "
              f"fall_hit={sc['fire_ok']} excess={sc['excess']}")

**The FLOOR gate is taken: event precision = recall = 1.000, zero phantom events, zero swaps** —
the production temporal stack, fed clean post-detector observations, reproduces every scenario.
The flags the clean run *does* raise are findings, not scoring noise: the production anomaly
false-alarms on three fast controlled lie-downs (that's S2's business), and the fastest stunt
fall (`G4_88_04`, bbox centre jumping 0.175/frame — real kinematics) fragments the tracker even
on clean boxes.

Under stress the event level collapses (P 0.20 → 0.13 → 0.03 mild/med/heavy) — and the FP
breakdown says **100% fragments, 0 foreign**: the stack never invents a wrong activity, it
shatters the right ones. Who's guilty? Paired ablations:

In [ ]:
print("-" * 78)
print("S1 ablations @ STRESS med — attribute the degradation to a layer")
print("-" * 78)
med = STRESS["med"]
for name, kw in [
    ("baseline (prod)", {}),
    ("tracker=fix15.3", dict(tracker_kw=dict(center_relink_lowconf_id=True))),
    ("stab=hybrid sl12", dict(stab_kw=dict(sl_dwell=12))),
    ("no face (no id)", dict(proto=(med[0], med[1], 0, med[3], med[4]))),
    ("no posture flip", dict(proto=(med[0], med[1], med[2], med[3], 0.0))),
]:
    proto = kw.pop("proto", med)
    rows = evaluate(_d, _meta, proto, **kw)
    print(f"{name:16} {fmt(summarize(rows))}")

Event P/R is **completely invariant** to the tracker fix, the stabilizer, the face and the posture
noise — while `excess/min` swings 5.38 → 1.91 with the LJ3 fix (nicely cross-validating LJ3's
5.43 → 0.78 on an independent harness). Track fragmentation and *event* fragmentation are
decoupled here: typing happens while seated (the tracker holds), and what shatters episodes is the
occlusion bursts themselves outlasting the FSM's exit grace (5 frames = 0.33 s). So the suspect is
a constant nobody had ever measured:

In [ ]:
print("-" * 78)
print("S1b exit-dwell sweep — FSM grace vs occlusion bursts (prod = 5/8)")
print("-" * 78)
for name, kw in [("prod 5/8", None), ("8/11", dict(exit_dwell=8, exit_young=11)),
                 ("10/13", dict(exit_dwell=10, exit_young=13)),
                 ("12/15", dict(exit_dwell=12, exit_young=15))]:
    for sname in ("med", "heavy"):
        rows = evaluate(_d, _meta, STRESS[sname], journal_kw=kw)
        s = summarize(rows)
        pci, rci = ci_prf(rows)
        print(f"exit={name:9} {sname:5} {fmt(s, pci, rci)}")

Monotone and large: raising the exit grace from 5/8 to 12/15 frames lifts event precision at the
live-like operating point from **0.133 to 0.667** (heavy: 0.029 → 0.545), by bridging exactly the
gaps that were splitting episodes. The cost is honest and small — an episode's end is committed
~0.5 s later. This is a 15.6 port candidate, licensed by measurement (and it keeps the
`> object_cadence` invariant the constant was originally chosen for).

One more stress casualty deserves its own act: the **fall alarm**.

## 4b. Numbers — S2: does the alarm know a fall from lying down?

The production rule (`anomaly.rs`): *a body that was standing within the last 3 s and reads lying
now* — keyed per track id, velocity-free. LJ2's E2 rule: *reached the floor* (min pelvis ≤ 0.35 m)
*AND peak downward pelvis velocity ≥ 1.13 m/s* (calibrated at 60 fps). The simulator can finally
race them on identical runs — including two variants the failures below motivate: an **id-free**
production rule (scene-level standing memory) and E2 **recalibrated to the production cadence**.

Under stress, why does the per-id rule lose falls? Trace the critical moment:

In [ ]:
s1c_g4_diagnostics(_d, _meta)
print()
s2_table(_d, _meta)

Three verdicts, all measured:

1. **The per-id production rule is broken on both sides.** On clean data it false-alarms on 3/10
   controlled floor scenarios — a person who lies down *briskly* (standing → sitting → lying in
   0.9–2.4 s) is inside the 3 s window; the LJ2 stabilizer veto can't help because these
   transitions legitimately pass through sitting. Under stress it drops to 4/6 hits — and the
   trace shows why: **the fall itself churns the track id** (the impact occludes the detector for
   a few frames; the bbox collapses faster than the re-link's centre gate), and the new id "never
   stood". The standing memory dies with the track exactly when it's needed.
2. **E2 at its 60 fps threshold misses one fall at 15 fps** — `G4_85_15` peaks at 1.12 m/s against
   the 1.13 threshold. The production cadence smears the peak; the threshold does not transfer
   as-is.
3. **E2 recalibrated to the cadence (≥1.05 m/s), scene-level: 0/10 false alarms + 6/6 hits, on
   FLOOR *and* under stress.** The honest caveat: the margin is thin (the worst quiet scenario
   peaks at 0.99, the weakest fall at 1.12), and CMU falls are stunt-performed — the threshold
   bounds a clinical fall from above (LJ2's caveat, still standing). **Ships to 15.6**: the E2
   physics rule, id-robust, cadence-recalibrated; the per-id window rule retires.

## 4c. Numbers — S3: which activities can data even supervise?

The raw-activity vocabulary (`typing`, `using_mouse`, `using_phone`, `holding_drink`, `reading`,
`watching_tv`) was an artifact of early eyeballed model play. OpenImages VRD is the only large
corpus with *human-labeled* visual relationships — below we count **per-image instances**
(val+test splits, ~110k rows), not the triplet vocabulary (which only says a pair is *possible*).

In [ ]:
# fetch the VRD metadata (public GCS, ~12 MB) into the notebook workspace
import urllib.request
os.makedirs("oi_meta", exist_ok=True)
for f in ["oidv6-validation-annotations-vrd.csv", "oidv6-test-annotations-vrd.csv",
          "oidv6-class-descriptions.csv"]:
    dst = os.path.join("oi_meta", f)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f"https://storage.googleapis.com/openimages/v6/{f}", dst)
        print("fetched", f)

In [ ]:
#!/usr/bin/env python3
"""S3 (LJ4): quantitative audit of the MVM raw-activity vocabulary against OpenImages
VRD — per-image relationship INSTANCES (val+test splits, human-labeled), not just the
triplet vocabulary of openimages_relations.py.

Directive (15.4): the current interaction_label map (fusion.rs:213 — typing/using_mouse/
using_phone/holding_drink/reading/watching_tv) is an eyeballed artifact of early model
play, not a data-driven vocabulary. This audit measures, for every current label and
every candidate, HOW MANY human-labeled instances the only large relationship corpus can
actually supervise — the input for the vocabulary-v2 verdict table (code change = 15.6).

Sources (oi_meta/, GCS public):
  oidv6-{validation,test}-annotations-vrd.csv   per-image instances (~110k rows)
  oidv6-class-descriptions.csv                  MID -> readable name

Emits kaggle-portfolio/research/LJ4-VRD-AUDIT.md.

Run:  ../venv/bin/python3 spike_pose/lj4_vrd_audit.py
"""

SPLITS = ["oidv6-validation-annotations-vrd.csv", "oidv6-test-annotations-vrd.csv"]
PERSON_NAMES = {"Person", "Man", "Woman", "Boy", "Girl", "Human body"}

# Current MVM raw-activity vocabulary (fusion.rs:213) -> the object classes that
# trigger it. VRD names differ from YOLO names (Computer keyboard vs keyboard).
MVM_VOCAB = {
    "typing":         ["Computer keyboard", "Laptop"],
    "using_mouse":    ["Computer mouse"],
    "using_phone":    ["Mobile phone", "Corded phone", "Telephone"],
    "holding_drink":  ["Coffee cup", "Mug", "Bottle", "Wine glass", "Bowl",
                       "Beer", "Wine", "Juice", "Coffee", "Tea", "Cocktail", "Milk"],
    "reading":        ["Book"],
    "watching_tv":    ["Television", "Remote control"],
}
# Relational/contextual labels with no object-class trigger (geometry-driven).
GEOMETRIC = ["people_together", "seated_at_workstation", "partially_in_frame"]


def load_names():
    names = {}
    with open(os.path.join(META, "oidv6-class-descriptions.csv"), newline="") as fh:
        for row in csv.reader(fh):
            if len(row) >= 2:
                names[row[0]] = row[1]
    return names


def load_instances(names):
    """-> list of (subject_name, object_name, predicate) per annotated instance."""
    rows = []
    for split in SPLITS:
        with open(os.path.join(META, split), newline="") as fh:
            for r in csv.DictReader(fh):
                rows.append((names.get(r["LabelName1"], r["LabelName1"]),
                             names.get(r["LabelName2"], r["LabelName2"]),
                             r["RelationshipLabel"]))
    return rows


def main():
    names = load_names()
    rows = load_instances(names)
    person_rows = [(s, o, p) for s, o, p in rows if s in PERSON_NAMES]
    print(f"VRD instances: {len(rows)} total, {len(person_rows)} person-subject "
          f"(val+test, human-labeled)")

    pred = collections.Counter(p for _, _, p in person_rows)
    pair = collections.Counter((p, o) for _, o, p in person_rows)
    is_states = collections.Counter(o for _, o, p in person_rows if p == "is")

    # Current vocabulary: instances per label = person rows whose object triggers it,
    # by predicate (holds / interacts_with dominate physical contact).
    lines = []
    lines.append("# LJ4 VRD audit — can data supervise the raw-activity vocabulary? (S3)")
    lines.append("")
    lines.append("> Auto-generated by `spike_pose/lj4_vrd_audit.py` from OpenImages V6 VRD")
    lines.append(f"> val+test instances ({len(rows)} rows, {len(person_rows)} person-subject).")
    lines.append("> Instances = human-labeled per-image relationship annotations, NOT the")
    lines.append("> triplet vocabulary (which only says a pair is *possible*).")
    lines.append("")
    lines.append("## Current MVM labels (fusion.rs:213) — supervisable instances")
    lines.append("")
    lines.append("| MVM label | trigger classes (VRD names) | instances | predicates seen |")
    lines.append("|---|---|---|---|")
    total_cur = 0
    for label, classes in MVM_VOCAB.items():
        cnt = collections.Counter()
        for s, o, p in person_rows:
            if o in classes:
                cnt[p] += 1
        n = sum(cnt.values())
        total_cur += n
        preds = ", ".join(f"{p}:{c}" for p, c in cnt.most_common()) or "—"
        lines.append(f"| `{label}` | {', '.join(classes)} | **{n}** | {preds} |")
    lines.append("")
    lines.append(f"Current 6 contact labels together: **{total_cur}** supervisable instances.")
    lines.append("")
    lines.append("Geometric labels (`" + "`, `".join(GEOMETRIC) + "`) have no VRD predicate —")
    lines.append("they are derived from scene geometry (3D distance / occlusion context) and")
    lines.append("need no external supervision. `at` (Desk/Table, "
                 f"{pair.get(('at', 'Desk'), 0) + pair.get(('at', 'Table'), 0)} instances) "
                 "corroborates `seated_at_workstation`.")
    lines.append("")
    lines.append("## Person-subject predicates by instance count (the add-candidate surface)")
    lines.append("")
    lines.append("| predicate | instances | top objects | MVM verdict input |")
    lines.append("|---|---|---|---|")
    HINT = {
        "holds": "carries the current contact family",
        "wears": "attribute, out of scope (clothing)",
        "interacts_with": "generic contact — the fusion edge itself",
        "ride": "NOT modeled: riding (bicycle/horse/skateboard)",
        "at": "corroborates seated_at_workstation",
        "on": "spatial context (sitting on chair/bench)",
        "plays": "NOT modeled: playing_instrument",
        "eat": "NOT modeled: eating",
        "drink": "drinking — holding_drink's honest big brother",
        "is": "STATE labels: posture/gait supervision, not an activity",
        "inside_of": "spatial context (vehicle)",
        "hug": "person-person contact — people_together refinement",
        "kiss": "person-person contact",
        "holding_hands": "person-person contact",
        "dance": "temporal-ish person activity",
        "talk_on_phone": "matches using_phone semantics directly",
        "read": "matches reading directly",
        "catch": "temporal action (needs video)",
        "throw": "temporal action (needs video)",
        "hits": "temporal action (needs video)",
        "kick": "temporal action (needs video)",
        "skateboard": "sport activity", "surf": "sport activity", "ski": "sport activity",
        "highfive": "person-person contact", "handshake": "person-person contact",
        "cut": "temporal action", "snowboard": "sport activity",
        "contain": "object-object", "hang": "spatial", "under": "spatial",
    }
    for p, n in pred.most_common():
        top = collections.Counter(o for _, o, pp in person_rows if pp == p)
        tops = ", ".join(f"{o} ({c})" for o, c in top.most_common(4))
        lines.append(f"| `{p}` | **{n}** | {tops} | {HINT.get(p, '—')} |")
    lines.append("")
    lines.append("## `is` state labels (posture/gait supervision — feeds the pose track, not FSM)")
    lines.append("")
    lines.append("| state | instances |")
    lines.append("|---|---|")
    for o, n in is_states.most_common():
        lines.append(f"| {o} | {n} |")
    lines.append("")

    lines.append("")
    lines.append("## Vocabulary v2 — the verdict table (S3 output; code change lands in 15.6)")
    lines.append("")
    lines.append("The mechanism (wrist/hand contact, object-major attribution) survives —")
    lines.append("it is unit-tested geometry. What the data refutes is the NAMES: specific")
    lines.append("activity claims (typing/watching_tv) that no corpus can supervise and that")
    lines.append("over-promise semantics a wrist-in-bbox test cannot deliver.")
    lines.append("")
    lines.append("| label | VRD instances | verdict |")
    lines.append("|---|---|---|")
    holds_n = pred.get("holds", 0)
    at_n = sum(c for (p, o), c in pair.items() if p == "at" and o in ("Table", "Desk"))
    verdicts = [
        ("typing", "0", "**fold** into `using_object(keyboard/laptop)` — name unsupervisable"),
        ("using_mouse", "0", "**fold** into `using_object(mouse)`"),
        ("using_phone", "15", "**fold** into `using_object(phone)` (VRD `talk_on_phone` = 0)"),
        ("holding_drink", "145", "**keep** — the one specialization with real support"),
        ("reading", "24", "**fold** into `using_object(book)`"),
        ("watching_tv", "0", "**drop** — wrist-on-remote/tv is not 'watching'; watching is "
                             "head-facing, a different signal entirely"),
        ("people_together", "— (geometric)", "**keep** — needs no supervision"),
        ("seated_at_workstation", f"{at_n} (`at` Table/Desk)", "**keep** — corroborated"),
        ("partially_in_frame", "— (honesty flag)", "**keep**"),
        ("`using_object(class)` umbrella", f"{holds_n} (`holds`)",
         "**add** — the data-supported claim IS 'person holds/uses X'; the class rides "
         "as a parameter, not a fabricated verb"),
        ("ride / plays / sports family", f"{pred.get('ride', 0)} / {pred.get('plays', 0)} / ~300",
         "**defer to phase 16** — supervisable but out-of-domain for a room camera"),
    ]
    for label, n, verdict in verdicts:
        lines.append(f"| `{label}` | {n} | {verdict} |")
    lines.append("")
    lines.append("Note: `is` carries **no lying state** — CMU mocap remains the only")
    lines.append("lying/fall supervision (the reference-pair complementarity, re-confirmed).")
    lines.append("`is Walking/Running` (" + f"{is_states.get('Walking', 0)}/{is_states.get('Running', 0)}"
                 + ") can weakly supervise `motion_state`, a pose-track asset, not FSM.")

    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    with open(OUT, "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print(f"wrote {OUT}")

    # Console: the verdict-relevant core.
    print("\ncurrent labels:")
    for label, classes in MVM_VOCAB.items():
        n = sum(1 for s, o, p in person_rows if o in classes)
        print(f"  {label:16} {n:6d} instances")
    print("\ntop person-subject predicates:")
    for p, n in pred.most_common(14):
        print(f"  {p:16} {n:6d}")
    print("\n'is' states:", dict(is_states.most_common(8)))



import csv
META = "oi_meta"
OUT = "./LJ4-VRD-AUDIT.md"
main()

The directive survives contact with the data — dramatically. Of the six shipped labels, three
have **zero** human-labeled instances (`typing`, `using_mouse`, `watching_tv`), two are near-zero
(`using_phone` 15, `reading` 24); only `holding_drink` (145) has real support. Meanwhile the
predicates the vocabulary *ignores* carry the mass: generic `holds` (2 153), `at` Table/Desk
(1 360 — corroborating `seated_at_workstation`), `ride` (1 432), `plays` (618). And the `is`
states hold a quiet treasure for other tracks: Standing 39 683 / Sitting 14 529 / **Walking 3 602 /
Running 2 156** (gait supervision for `motion_state`) — but **no lying state at all**: CMU mocap
remains the only fall/lying source, re-confirming the reference-pair complementarity from LJ0.

**Vocabulary v2 verdict** (code change lands in 15.6 as a documented contract change): keep the
*mechanism* (wrist/hand contact, object-major attribution — it's unit-tested geometry), fix the
*names*. Fold `typing`/`using_mouse`/`using_phone`/`reading` into a data-supportable
`using_object(class)` umbrella; **drop** `watching_tv` (a wrist on the remote is not "watching" —
watching is head-facing, a different signal); keep `holding_drink` and the geometric labels
(`people_together`, `seated_at_workstation`, `partially_in_frame`). The sports/riding family is
supervisable but out-of-domain for a room camera — deferred to phase 16.

## 5. Verdict

| hypothesis | outcome |
|---|---|
| S1: FLOOR P/R = 1.0, zero phantoms | **confirmed** — the gate is taken |
| S1: stress degradation attributable | **confirmed** — 100% fragments; invariant to tracker/stabilizer/face/flip; the knob is the FSM exit grace (0.133 → 0.667 @ 12/15) |
| S2: prod fall rule flawed | **confirmed twice** — 3/10 false alarms on clean lie-downs; id-churn loses 2/6 falls under stress |
| S2: E2 transfers | **partially** — 60 fps threshold misses one fall at 15 fps; recalibrated (≥1.05) it is clean on both axes, thin margin stated |
| S3: vocabulary is an artifact | **confirmed by data** — 3 labels at zero instances; verdict table shipped |

And one meta-result: **the simulator caught its own compiler twice** (the settling-frame phantom
fall, the walk-in typing contact) before it ever tested the product — exactly the discipline the
failure criteria demanded.

## 6. What it changes — and the bridge to LJ5

Into the 15.6 port package, each item licensed by a number in this journal:

1. **Fall rule**: retire the per-id 3 s window; ship the E2 physics rule (floor + pelvis descent),
   **scene-level** (id-robust — the fall itself churns ids), threshold recalibrated at the
   production cadence.
2. **FSM exit grace**: raise from 5/8 (a constant chosen for YOLO cadence flicker, never measured
   against occlusion bursts) toward 10–12/13–15 — event precision at live-like noise triples.
3. **Vocabulary v2**: `using_object(class)` umbrella, `watching_tv` dropped — a documented
   StateFile/MCP contract change, alongside LJ1's flexion channel.
4. **Golden scenarios in CI, permanently**: a 6-instance FLOOR subset now runs as a model-free
   `cargo test` inside the Rust workspace (the same compiled file this notebook scores), pinning
   today's contract; the asserts that these findings flip are marked for 15.6.

What the simulator does *not* cover, honestly: the pixel detectors (out of scope by design), the
head-on full-occlusion id swap (inherited from LJ3, characterized here on G5-STRESS, still open),
and clinical fall kinematics (CMU's falls are stunts; the threshold is an upper bound).

Next, **LJ5 — Weighing the experts**: with the logic layers measured, what do the ONNX experts
actually cost — latency, bytes, and honest public-bench accuracy — and what can the 15.1 results
already delete?

In [ ]:
# ---- passport: bind every number to the exact data (+ code commit when run from the repo) ----
import hashlib, subprocess
def sha16(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""):
            h.update(ch)
    return h.hexdigest()[:16]
try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                     stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    commit = "(kaggle run — no git)"
passport = {"code_commit": commit, "golden_scenarios": sha16(GOLD)}
print("passport:", passport)